# 과일 선화 인식 CNN: 외곽선과 내부 특징의 효과 비교

**분류 대상:** 사과 · 바나나 · 레몬 · 딸기 · 파인애플  
**연구 질문:** 같은 CNN에 외곽선만 입력할 때와 내부 특징을 포함한 선화를 입력할 때 성능은 얼마나 달라지는가?

사진을 그림으로 바꾼 뒤 CNN으로 분류하는 프로젝트이다. 색상 사진을 직접 분류하는 CNN이나 사람이 그린 그림의 성능을 검증한 실험은 아니다. 여기서 선화는 외곽선과 보이는 내부 명암 경계·무늬를 이진 선으로 표현한 그림이며, 가려진 부분을 복원하지 않는다.

## 실행 전 확인

- VS Code에서 **이 노트북이 있는 프로젝트 폴더**를 열고 로컬 Python 커널을 선택한다. 개발·실험 환경은 Python 3.12.6이며 `.venv` 커널을 사용했다.
- 설치 셀은 처음 한 번 실행한다. 설치·업데이트 후에는 커널을 재시작하고 **0-B 설정**부터 실행한다. 설치 셀 실행과 커널 재시작을 한 번의 Run All로 처리할 수는 없다.
- 저장 경로는 `Path.cwd() / 'fruit_outline_project'`이다. 설정 셀의 저장 폴더 출력이 의도한 위치인지 확인한다. 모델과 이미지는 노트북 파일 안에 포함되지 않으며 최초 다운로드에는 인터넷이 필요하다.
- 두 CNN 학습과 모델 다운로드에는 시간이 걸린다. 기존 실험 환경에서는 CNN 학습이 각각 약 3분이었지만 PC·패키지·네트워크에 따라 달라진다. 자동 분리 모델의 최초 다운로드 시간은 별도이다.

## 목적별 실행 순서

| 목적 | 실행할 셀 | 종료 지점 |
|---|---|---|
| **핵심 실험 재현** | 0-B → 1 → 2의 함수와 미리보기 → 3 → 4-A → 4-B | 두 CNN의 성능 비교 완료 |
| 저장된 비교 모델로 보고서 그래프 재생성 | 0-B → 2의 **함수 셀만** → 5 → 4-B | 사진·모델과 비교 실험 폴더가 이미 있을 때 가능 |
| 저장된 모델로 새 사진 분석 | 0-B → 2의 **함수 셀만** → 5 → 6-A → 6-B | 6-B 실행 즉시 파일 선택 창 표시 |
| 현재 선화 모델의 회전 평균 추가 분석 | 5 실행 후 5-A → 5-B | 선택 기능. 4-B에도 회전 평균 결과가 있음 |
| 사람의 정답으로 추가 학습 | 6-B → 7-A → 7-B | 선택 기능. 기본 비교 실험에 포함하지 않음 |

첫 실행은 핵심 실험 순서로 진행한다. **과제의 두 CNN 비교는 4-B에서 끝나므로 7번 정답 교정은 실행할 필요가 없다.** 전체 셀을 연속 실행하면 6-B 사진 선택 창과 7-A 정답 교정 창에서 입력을 기다리고, 7-B에서 저장된 교정 자료로 추가 학습할 수 있으므로 제출 확인은 위 실행 경로를 따라 진행한다.

저장된 결과를 사용할 때는 같은 프로젝트의 데이터·모델·결과 폴더도 필요하다. 다른 PC로 옮기면 저장된 목록의 절대 경로가 맞지 않을 수 있으므로 최초에는 1~4번을 재실행해 현재 위치에 결과를 생성한다. 2번 미리보기는 1번의 다운로드 목록이 있어야 실행된다.

## 비교 설계와 범위

두 모델은 원본 분할, 마스크, 증강본, 정답, 96×96×1 입력 크기, CNN 구조, 초기 가중치, 에폭별 순서, 배치 크기, 학습률, 총 에폭 수와 검증 모델 선택 규칙을 공유한다. **비교 요인은 내부 선의 포함 여부**이다. 같은 횟수만큼 학습한 뒤 각각 검증 손실이 최소인 가중치를 선택한다. 고정된 주 실험에는 조기 종료나 모델별 학습률 감소를 사용하지 않는다.

주 비교는 두 모델 모두 **회전 없는 0도 단일 예측**이다. 4방향 점수 평균은 별도 보조 실험으로 기록한다. 시드가 하나이면 반복 변동성을 판단할 수 없으며, `EXPERIMENT_SEEDS`에 여러 값을 지정하면 반복 실험을 진행하지만 실행 시간이 늘어난다.

Grounding DINO와 SlimSAM은 사진의 과일 탐지·분리를 위해 **사전학습 가중치를 불러와 추론에만 활용**한다. 직접 학습·비교하는 대상은 두 CNN이다. 탐지한 과일 이름을 CNN의 최종 답으로 쓰지 않는다. 공개 테스트의 분류 지표와 실사진 탐지·분리·분류 전체 성능은 구분한다.


## 0-A. 실행 환경 설치

최초 실행 또는 패키지 변경 시에만 실행한다. 이미 설치된 환경에서는 건너뛰고 0-B로 이동한다.


In [ ]:
%pip install -q "tensorflow>=2.16,<2.22" opencv-python-headless pillow matplotlib pandas scikit-learn "torch>=2.6,<3" torchvision "transformers>=4.46,<5" safetensors


설치가 완료되면 VS Code의 **Restart Kernel(커널 재시작)**을 실행하고 아래 0-B부터 진행한다. 설치 출력에 명령어나 진행 로그가 보이는 것만으로 오류는 아니다. `ERROR`, 설치 실패 또는 traceback이 있으면 해당 오류를 먼저 확인한다.

TensorFlow는 CNN 학습·예측, PyTorch/Transformers는 자동 탐지·분리에 사용한다. 사진 선택 창은 로컬 `tkinter`를 사용하므로 원격·화면 없는 커널에서는 열리지 않을 수 있다. 설치 셀은 최초 또는 의존성이 바뀐 경우에만 실행한다.


## 0-B. 공통 설정과 라이브러리

모든 실행 경로의 시작점이다. 아래 기본값이 PPT 비교 실험의 조건이며, 개인 PC의 절대 경로를 코드에 직접 지정하지 않는다.


In [ ]:
# 0-B 설정: 경로·대상·비교 조건을 정합니다. PPT 재현 시 기본 학습 조건을 유지하세요.
import os, io, json, time, hashlib, re
from pathlib import Path
from datetime import datetime
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.request import Request, urlopen
from urllib.parse import quote
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

ROOT = Path.cwd() / 'fruit_outline_project'
DATA_DIR = ROOT / 'photos'
RESULT_DIR = ROOT / 'results' / 'lineart_v1_apple_banana_lemon_strawberry_pineapple'
CONTROLLED_ROOT = ROOT / 'results' / 'controlled_outline_vs_lineart_v1'
CONTROLLED_ROOT.mkdir(parents=True,exist_ok=True)
CONTROLLED_REGISTRY = CONTROLLED_ROOT / 'latest_experiment.json'
EXPERIMENT_SEEDS = [42]  # 보고서용 반복 실험: [42, 123, 2026]
for folder in [DATA_DIR, RESULT_DIR]: folder.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ['apple', 'banana', 'lemon', 'strawberry', 'pineapple']
CLASS_KO = ['사과', '바나나', '레몬', '딸기', '파인애플']
SEED = 42
TRAIN_PER_CLASS = 450  # 빠른 작동 확인: 60 / 충분히 검증한 뒤 수량을 늘리세요
TEST_PER_CLASS = 100
MAX_SOURCE_FOLDERS = 3  # 가능한 품종/촬영 폴더를 최대 3개씩 선택
IMAGE_SIZE, EPOCHS, BATCH_SIZE = 96, 25, 32
REPRESENTATION = 'fruit_lineart_v1'
MODEL_METADATA = None
MODEL_TO_LOAD = 'latest'  # 'original'로 바꾸면 최초 모델 사용
MODEL_REGISTRY = RESULT_DIR / 'latest_finetuned.json'
FEEDBACK_DIR = ROOT / 'reviewed_lineart'
FEEDBACK_INDEX = FEEDBACK_DIR / 'labels.json'
FINETUNE_DIR = RESULT_DIR / 'finetuned_versions'
ACTIVE_MODEL_PATH = RESULT_DIR / 'fruit_lineart_cnn.keras'
LAST_SCENE_DIR = None
MASK_CONFIG = dict(white_v=185, white_s=75, min_area_fraction=0.003)
LINEART_CONFIG = dict(margin=10, line_width=1, canny_low=35, canny_high=100,
                     adaptive_block=11, adaptive_c=5, bilateral_d=5, bilateral_sigma=35,
                     min_component_pixels=2)
AUGMENTED_VIEWS = 2  # 학습 원본마다 추가할 변형 수. 검증/테스트에는 추가하지 않음
OCCLUSION_PROBABILITY = 0.35
MAX_OCCLUSION_FRACTION = 0.20
# 자동 분리 모델과 GPU를 함께 사용할 때 TensorFlow가 메모리를 모두 선점하지 않도록 설정
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| OpenCV:', cv2.__version__)
print('저장 폴더:', ROOT)


## 1. 선택한 과일 사진만 다운로드
전체 저장소 ZIP 대신 필요한 이미지의 목록을 GitHub API로 읽어 선택한 파일만 다운로드합니다. 최초 실행은 인터넷 연결이 필요하고 몇 분 이상 걸릴 수 있습니다. 목록·사진은 캐시하여 재사용합니다.

첫 실행의 저장소 커밋을 고정하므로 같은 프로젝트 폴더에서는 동일 버전을 사용합니다. GitHub API 사용량 제한(HTTP 403/429)이 나오면 나중에 재실행하거나 `GITHUB_TOKEN` 환경변수에 본인 토큰을 설정할 수 있습니다. 토큰은 출력/저장하지 않습니다.

사과·바나나·레몬·딸기·파인애플의 폴더를 확인하여 최대 3개씩 선정합니다. 온전한 딸기를 사용하도록 잘린 딸기인 `Strawberry Wedge`는 제외합니다. 아래 표에 실제 사용한 폴더가 표시됩니다.

**이 셀의 결과:** 선택한 품종/촬영 폴더와 사진 목록 `manifest`를 만들고 필요한 파일을 `photos`에 저장한다. 다운로드가 끝나면 2번에서 실제 마스크·선화를 먼저 확인한다. 기본 설정의 과일별 450장은 공식 Training에서 가져오는 총량이며, 이후 3번에서 학습과 검증으로 다시 나눈다.


In [ ]:
# 1 다운로드: 버전 고정 → 품종 폴더 선택 → 사진 목록 → 필요한 사진 저장.
REPO = 'fruits-360/fruits-360-100x100'
API = f'https://api.github.com/repos/{REPO}'
META_DIR = ROOT / 'metadata'
META_DIR.mkdir(exist_ok=True)

def github_json(endpoint):
    cache = META_DIR / (hashlib.sha256(endpoint.encode()).hexdigest() + '.json')
    if cache.exists(): return json.loads(cache.read_text(encoding='utf-8'))
    headers = {'User-Agent': 'FruitOutlineNotebook', 'Accept': 'application/vnd.github+json'}
    token = os.environ.get('GITHUB_TOKEN')
    if token: headers['Authorization'] = f'Bearer {token}'
    try:
        with urlopen(Request(API + endpoint, headers=headers), timeout=60) as response:
            data = json.load(response)
    except Exception as exc:
        raise RuntimeError('GitHub 목록 조회 실패. 인터넷/API 제한을 확인하고 재실행하세요.') from exc
    cache.write_text(json.dumps(data), encoding='utf-8')
    return data

version_file = META_DIR / 'version.json'
if version_file.exists():
    COMMIT = json.loads(version_file.read_text())['commit']
else:
    COMMIT = github_json('/commits/main')['sha']
    version_file.write_text(json.dumps({'commit': COMMIT}), encoding='utf-8')

def tree(sha):
    result = github_json(f'/git/trees/{sha}')
    if result.get('truncated'): raise RuntimeError('파일 목록이 잘렸습니다. 전체 목록으로 진행할 수 없습니다.')
    return result['tree']

root_entries = {e['path']: e for e in tree(COMMIT)}
folder_entries = {split: {e['path']: e for e in tree(root_entries[split]['sha']) if e['type'] == 'tree'}
                  for split in ['Training', 'Test']}
prefixes = {'apple':'Apple ', 'banana':'Banana ', 'lemon':'Lemon ', 'strawberry':'Strawberry ', 'pineapple':'Pineapple '}
preferred = {'apple':['Apple Red 1', 'Apple Golden 1'], 'banana':['Banana 1', 'Banana Lady Finger 1'],
             'lemon':['Lemon 1', 'Lemon Meyer 1'], 'strawberry':['Strawberry 1', 'Strawberry 2'],
             'pineapple':['Pineapple 1', 'Pineapple Mini 1']}
source_folders = {}
for label in CLASS_NAMES:
    common = set(folder_entries['Training']) & set(folder_entries['Test'])
    candidates = sorted(n for n in common if n.startswith(prefixes[label])
                        and not (label == 'strawberry' and 'Wedge' in n))
    ordered = [n for n in preferred[label] if n in candidates]
    ordered += [n for n in candidates if n not in ordered]
    if not ordered: raise ValueError(f'{label}의 사진 폴더를 찾지 못했습니다. 데이터셋 폴더를 확인하세요.')
    source_folders[label] = ordered[:MAX_SOURCE_FOLDERS]
display(pd.DataFrame([{'class': c, 'source_folder': f} for c, fs in source_folders.items() for f in fs]))

def choose_evenly_across_folders(groups, limit, rng):
    groups = [list(rng.permutation(g)) for g in groups]
    selected = []
    for i in range(max(map(len, groups))):
        for group in groups:
            if i < len(group): selected.append(group[i])
            if len(selected) >= limit: return selected
    return selected

records = []
for split, limit in [('Training', TRAIN_PER_CLASS), ('Test', TEST_PER_CLASS)]:
    for class_id, label in enumerate(CLASS_NAMES):
        groups = []
        for folder in source_folders[label]:
            entries = tree(folder_entries[split][folder]['sha'])
            files = [f'{split}/{folder}/{e["path"]}' for e in entries
                     if e['type']=='blob' and Path(e['path']).suffix.lower() in ['.jpg','.jpeg','.png']]
            if not files: raise ValueError(f'사진이 없습니다: {split}/{folder}')
            groups.append(sorted(files))
        selected = choose_evenly_across_folders(groups, limit, np.random.default_rng(SEED+class_id))
        for relative in selected:
            records.append({'official_split':split, 'class':label, 'label':class_id,
                            'source_folder':relative.split('/')[1], 'source_path':relative,
                            'local_path':str(DATA_DIR / relative)})

def download_photo(record):
    target = Path(record['local_path'])
    if target.exists():
        try:
            with Image.open(target) as im: im.verify()
            return
        except OSError: target.unlink()
    target.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://raw.githubusercontent.com/{REPO}/{COMMIT}/' + quote(record['source_path'], safe='/')
    for attempt in range(3):
        try:
            with urlopen(Request(url, headers={'User-Agent':'FruitOutlineNotebook'}), timeout=45) as response:
                content = response.read()
            with Image.open(io.BytesIO(content)) as im: im.verify()
            target.write_bytes(content)
            return
        except Exception:
            if attempt == 2: raise
            time.sleep(attempt+1)

errors = []
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(download_photo, row):row for row in records}
    for done, future in enumerate(as_completed(futures), 1):
        try: future.result()
        except Exception as exc: errors.append({'file':futures[future]['source_path'], 'error':str(exc)})
        if done % 100 == 0 or done == len(records): print(f'다운로드/캐시 확인: {done}/{len(records)}')
if errors:
    display(pd.DataFrame(errors))
    raise RuntimeError('일부 다운로드가 실패했습니다. 이 셀을 다시 실행하면 기존 파일은 재사용합니다.')
manifest = pd.DataFrame(records)
manifest.to_csv(CONTROLLED_ROOT / 'source_manifest_latest.csv', index=False, encoding='utf-8-sig')
print('저장소 커밋:', COMMIT)
display(pd.crosstab(manifest['class'], manifest['official_split']))


## 2. 공통 선화 생성 함수와 미리보기
학습과 실사진 모두 `fruit_to_lineart(RGB, 물체 마스크)`로 변환합니다.
배경 제거 → 동일 비율/크기로 정규화 → 흑백 변환 → 경계 보존 필터 → Canny와 적응형 이진화로 내부 선 추출 → 외곽선과 결합.
CNN은 완성된 이진 선화만 봅니다. 색상과 원본 명암 영상은 입력하지 않습니다.
아래 미리보기에서 **외곽선만 있는 그림과 내부 선까지 있는 그림**을 비교하세요. `LINEART_CONFIG`를 바꾸면 재학습해야 합니다.

**코드 읽는 순서:** `read_rgb`는 사진 읽기, `clean_object_mask`는 마스크 정리, `training_object_mask`는 흰 배경의 학습 사진 분리, `fruit_to_lineart`는 그림 생성, `augment_training_object`는 학습용 변형을 담당한다. 아래의 점수 평균·사진 식별 함수는 이후 사진 분석과 교정 기능에서 재사용한다.

`training_object_mask`는 Fruits-360처럼 흰 배경인 학습 사진용이다. 복잡한 실사진에는 이 함수를 그대로 적용하지 않고 6-A의 사전학습 탐지·분리 모델이 만든 마스크를 사용한다.


In [ ]:
# 2 함수 정의: 사진 읽기 → 마스크 정리 → 선화 생성 → 증강 → 예측 보조 함수.
def read_rgb(source):
    if isinstance(source,np.ndarray):
        if source.ndim!=3 or source.shape[2]!=3: raise ValueError('RGB 3채널 배열이 필요합니다.')
        return source.astype(np.uint8)
    with Image.open(source) as image:
        image=ImageOps.exif_transpose(image).convert('RGBA')
        white=Image.new('RGBA',image.size,(255,255,255,255))
        return np.asarray(Image.alpha_composite(white,image).convert('RGB'))

def clean_object_mask(mask):
    binary=(np.asarray(mask)>0).astype(np.uint8)*255
    if binary.ndim!=2: raise ValueError('물체 마스크는 2차원이어야 합니다.')
    contours,_=cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not contours: raise ValueError('물체 영역이 없습니다.')
    contour=max(contours,key=cv2.contourArea)
    if cv2.contourArea(contour)<2: raise ValueError('물체 영역이 너무 작습니다.')
    clean=np.zeros_like(binary)
    cv2.drawContours(clean,[contour],-1,255,cv2.FILLED)
    return clean

def fruit_to_lineart(rgb,mask,image_size=96,margin=10,line_width=1,
                     canny_low=35,canny_high=100,adaptive_block=11,adaptive_c=5,
                     bilateral_d=5,bilateral_sigma=35,min_component_pixels=2):
    """학습과 실제 사진에 공통 적용: 외곽선 + 내부 명암 경계/무늬를 이진 선화로 변환."""
    rgb=read_rgb(rgb)
    clean=clean_object_mask(mask)
    if clean.shape!=rgb.shape[:2]: raise ValueError('사진과 마스크의 크기가 다릅니다.')
    if not 0<=margin<image_size/2: raise ValueError('선화 여백 설정이 잘못되었습니다.')
    if not 0<=canny_low<canny_high<=255: raise ValueError('Canny 기준값을 확인하세요.')
    if adaptive_block<3 or adaptive_block%2==0: raise ValueError('adaptive_block은 3 이상 홀수여야 합니다.')
    # 배경의 선이 CNN으로 전달되지 않도록 물체 외부를 먼저 흰색으로 치환
    isolated=rgb.copy(); isolated[clean==0]=255
    ys,xs=np.where(clean>0)
    x,y,w,h=int(xs.min()),int(ys.min()),int(xs.max()-xs.min()+1),int(ys.max()-ys.min()+1)
    ratio=(image_size-2*margin)/max(w,h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    left,top=(image_size-nw)//2,(image_size-nh)//2
    normalized_rgb=np.full((image_size,image_size,3),255,np.uint8)
    normalized_mask=np.zeros((image_size,image_size),np.uint8)
    interp=cv2.INTER_AREA if ratio<1 else cv2.INTER_CUBIC
    normalized_rgb[top:top+nh,left:left+nw]=cv2.resize(isolated[y:y+h,x:x+w],(nw,nh),interpolation=interp)
    normalized_mask[top:top+nh,left:left+nw]=cv2.resize(clean[y:y+h,x:x+w],(nw,nh),interpolation=cv2.INTER_NEAREST)
    normalized_rgb[normalized_mask==0]=255
    gray=cv2.cvtColor(normalized_rgb,cv2.COLOR_RGB2GRAY)
    smooth=cv2.bilateralFilter(gray,int(bilateral_d),float(bilateral_sigma),float(bilateral_sigma))
    canny=cv2.Canny(smooth,int(canny_low),int(canny_high),L2gradient=True)
    dark_details=255-cv2.adaptiveThreshold(smooth,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY,int(adaptive_block),float(adaptive_c))
    # 흰 배경과 만나는 경계 띠를 내부 특징으로 오인하지 않도록 제외
    radius=max(2,int(adaptive_block)//2)
    interior=cv2.erode(normalized_mask,np.ones((2*radius+1,2*radius+1),np.uint8),iterations=1)
    detail_edges=cv2.Canny(dark_details,50,150)
    internal=cv2.bitwise_and(cv2.bitwise_or(canny,detail_edges),interior)
    # 작은 점 잡음은 제거하되, 씨/껍질 무늬 등 연결된 작은 특징은 유지
    if min_component_pixels>1:
        count,labels,stats,_=cv2.connectedComponentsWithStats(internal,connectivity=8)
        keep=np.zeros(count,np.uint8)
        if count>1: keep[1:]=(stats[1:,cv2.CC_STAT_AREA]>=min_component_pixels).astype(np.uint8)*255
        internal=keep[labels]
    outline=np.zeros_like(normalized_mask)
    contours,_=cv2.findContours(normalized_mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(outline,contours,-1,255,int(line_width))
    lineart=cv2.bitwise_or(outline,internal)
    tensor=lineart.astype('float32')[...,None]/255.0
    details={'rgb':rgb,'mask':clean,'normalized_rgb':normalized_rgb,'normalized_mask':normalized_mask,
             'outline':outline,'internal':internal,'lineart':lineart,'drawing':255-lineart,
             'bbox':(x,y,w,h),'touches_border':bool(x==0 or y==0 or x+w==clean.shape[1] or y+h==clean.shape[0])}
    return tensor,details

def training_object_mask(rgb,white_v=185,white_s=75,min_area_fraction=0.003):
    hsv=cv2.cvtColor(rgb,cv2.COLOR_RGB2HSV)
    background=cv2.inRange(hsv,(0,0,int(white_v)),(179,int(white_s),255))
    mask=cv2.bitwise_not(background)
    mask=cv2.copyMakeBorder(mask,4,4,4,4,cv2.BORDER_CONSTANT,value=0)
    kernel=np.ones((3,3),np.uint8)
    mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,kernel)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel,iterations=2)[4:-4,4:-4]
    mask=clean_object_mask(mask)
    fraction=float((mask>0).mean())
    if fraction<min_area_fraction: raise ValueError('학습 사진의 물체 영역이 너무 작습니다.')
    if fraction>0.95: raise ValueError('학습 사진의 흰 배경 분리를 확인하세요.')
    return mask

def extract_training_lineart(source):
    rgb=read_rgb(source)
    mask=training_object_mask(rgb,**MASK_CONFIG)
    return fruit_to_lineart(rgb,mask,IMAGE_SIZE,**LINEART_CONFIG)

def augment_training_object(rgb,mask,rng):
    """학습 원본만 회전/반전/부분 가림 후 공통 선화 변환에 전달."""
    rgb=read_rgb(rgb);mask=(np.asarray(mask)>0).astype(np.uint8)*255
    pad=max(rgb.shape[:2])//2+4
    photo=cv2.copyMakeBorder(rgb,pad,pad,pad,pad,cv2.BORDER_CONSTANT,value=(255,255,255))
    visible=cv2.copyMakeBorder(mask,pad,pad,pad,pad,cv2.BORDER_CONSTANT,value=0)
    h,w=visible.shape
    matrix=cv2.getRotationMatrix2D((w/2,h/2),float(rng.uniform(-180,180)),1)
    photo=cv2.warpAffine(photo,matrix,(w,h),flags=cv2.INTER_LINEAR,borderValue=(255,255,255))
    visible=cv2.warpAffine(visible,matrix,(w,h),flags=cv2.INTER_NEAREST,borderValue=0)
    if rng.random()<0.5: photo=cv2.flip(photo,1);visible=cv2.flip(visible,1)
    if rng.random()<OCCLUSION_PROBABILITY:
        ys,xs=np.where(visible>0)
        if len(xs):
            x1,y1,x2,y2=int(xs.min()),int(ys.min()),int(xs.max())+1,int(ys.max())+1
            cut=visible.copy(); side=int(rng.integers(4)); amount=float(rng.uniform(0.05,MAX_OCCLUSION_FRACTION))
            if side==0: cut[:,:x1+max(1,round((x2-x1)*amount))]=0
            elif side==1: cut[:,x2-max(1,round((x2-x1)*amount)):]=0
            elif side==2: cut[:y1+max(1,round((y2-y1)*amount)),:]=0
            else: cut[y2-max(1,round((y2-y1)*amount)):,:]=0
            if np.count_nonzero(cut)>=0.60*np.count_nonzero(visible): visible=cut
    photo=np.clip(photo.astype('float32')*rng.uniform(0.9,1.1),0,255).astype('uint8')
    return photo,visible

def show_steps(details,title=''):
    fig,axes=plt.subplots(1,4,figsize=(12,3))
    for ax,key,label in zip(axes,['rgb','mask','outline','drawing'],
                            ['Photo','Object mask','Outline only','Line art: contour + details']):
        ax.imshow(details[key],cmap='gray',vmin=0,vmax=255);ax.set_title(label);ax.axis('off')
    fig.suptitle(title);plt.tight_layout();plt.show()

def ensure_lineart_model():
    metadata=globals().get('MODEL_METADATA')
    if not isinstance(metadata,dict) or metadata.get('representation')!=REPRESENTATION:
        raise RuntimeError('내부 특징 선화용 새 모델이 필요합니다. 1~4번에서 재학습한 뒤 5번 셀을 실행하세요.')
    if metadata.get('classes')!=CLASS_NAMES or metadata.get('image_size')!=IMAGE_SIZE:
        raise ValueError('모델의 과일 종류 또는 선화 크기가 현재 설정과 다릅니다.')
    if metadata.get('lineart_config')!=LINEART_CONFIG:
        raise ValueError('선화 설정이 학습 시점과 다릅니다. 설정을 복원하거나 새 설정으로 재학습하세요.')
    current=globals().get('model')
    if current is None or tuple(current.input_shape[1:])!=(IMAGE_SIZE,IMAGE_SIZE,1):
        raise ValueError('선화 CNN 입력 크기가 맞지 않습니다. 5번 모델 불러오기를 실행하세요.')

def predict_lineart_scores(tensors,batch_size=32,prediction_model=None):
    """같은 CNN으로 0/90/180/270도를 예측하고 종류별 softmax 점수를 동일 가중치로 평균."""
    ensure_lineart_model()
    batch=np.asarray(tensors,dtype=np.float32)
    if batch.size==0:
        empty=np.empty((0,len(CLASS_NAMES)),dtype=np.float32)
        return {'single':empty,'rotation_mean':empty.copy(),
                'by_rotation':np.empty((0,4,len(CLASS_NAMES)),dtype=np.float32)}
    if batch.ndim!=4 or batch.shape[1:]!=(IMAGE_SIZE,IMAGE_SIZE,1):
        raise ValueError('입력은 (N, IMAGE_SIZE, IMAGE_SIZE, 1) 형태의 선화만 지정하세요.')
    if not np.isfinite(batch).all() or batch.min()<0 or batch.max()>1:
        raise ValueError('선화의 픽셀값은 0~1이어야 합니다.')
    if not isinstance(batch_size,(int,np.integer)) or batch_size<1:
        raise ValueError('batch_size는 1 이상의 정수여야 합니다.')
    by_rotation=np.empty((len(batch),4,len(CLASS_NAMES)),dtype=np.float32)
    for start in range(0,len(batch),batch_size):
        chunk=batch[start:start+batch_size]
        # 90도 단위로 픽셀 위치만 바꾸며 보간하거나 선화를 다시 그리지 않습니다.
        rotated=np.ascontiguousarray(np.concatenate(
            [np.rot90(chunk,k,axes=(1,2)) for k in range(4)],axis=0))
        network=model if prediction_model is None else prediction_model
        prob=np.asarray(network.predict(rotated,batch_size=batch_size,verbose=0),dtype=np.float32)
        expected=(4*len(chunk),len(CLASS_NAMES))
        if prob.shape!=expected or not np.isfinite(prob).all():
            raise ValueError('CNN 출력 형태/값이 현재 5종 설정과 맞지 않습니다.')
        if (prob<0).any() or not np.allclose(prob.sum(axis=1),1,atol=1e-4):
            raise ValueError('softmax 점수를 반환하는 CNN을 사용하세요.')
        by_rotation[start:start+len(chunk)]=prob.reshape(4,len(chunk),len(CLASS_NAMES)).transpose(1,0,2)
    return {'single':by_rotation[:,0,:].copy(),
            'rotation_mean':by_rotation.mean(axis=1),'by_rotation':by_rotation}
def photo_fingerprint(rgb):
    pixels=np.ascontiguousarray(read_rgb(rgb))
    return hashlib.sha256(str(pixels.shape).encode()+pixels.tobytes()).hexdigest()
def active_audit_path():
    return Path(MODEL_METADATA.get('dataset_audit_path',RESULT_DIR/'preprocessing_audit.csv'))


In [ ]:
# 2 미리보기: 1번의 manifest가 있어야 합니다. 저장 모델만 시연할 때는 생략 가능합니다.
# 학습 전에 내부 무늬가 선화에 남는지 종류별로 확인합니다.
for label in CLASS_NAMES:
    example=manifest[(manifest['class']==label)&(manifest['official_split']=='Training')].iloc[0]
    try:
        tensor,details=extract_training_lineart(example['local_path'])
        show_steps(details,label)
        print(label,'내부 선 픽셀:',np.count_nonzero(details['internal']))
    except ValueError as exc:
        print(label,':',exc)


## 3. 같은 분할·증강에서 외곽선/선화 입력 쌍 구성
공식 `Test`는 최종 공개 평가에만 씁니다. 공식 `Training`의 각 종류에서 촬영/품종 폴더가 여러 개라면 한 폴더를 검증용으로 통째로 분리합니다. 폴더가 하나뿐이면 파일의 회전 시퀀스(`r`, `r2` 등)를 그룹으로 묶어 하나를 검증용으로 분리합니다. 그룹이 하나뿐인 경우 프레임 번호의 뒤 20%를 검증에 배정하고 경계의 5프레임은 제외합니다.

흰 배경 분리에 실패한 파일은 목록에 기록하고 제외합니다. 완전히 같은 원본 픽셀의 중복은 학습→검증→테스트 순으로 제거합니다. 종류별 최소 수량이 부족하면 중단합니다.

**주의:** Fruits-360은 회전 영상에서 만든 연속 프레임을 포함합니다. 이 분할이 과일 개체 수준의 독립성을 보장하지는 않습니다. 공개 정확도를 새로운 실제 과일에 대한 성능으로 해석하지 말고, 직접 촬영한 다른 과일로 별도 평가하세요.

학습·검증·테스트를 원본 사진 단위로 먼저 나눕니다. 그 후 학습 원본에만 ±180도 회전·반전과 확률적인 가벼운 부분 가림을 적용하고, 공통 함수로 선화를 다시 만듭니다. 검증/테스트에는 원본 선화만 사용합니다. 부분 가림 증강이 실제 겹침을 완전히 재현하지는 않습니다.

외곽선과 선화를 별도로 무작위 생성하지 않고 같은 사진/마스크/증강본에서 함께 생성합니다. 한 쪽만 전처리에 실패한 샘플을 따로 포함하지 않습니다. 분할 목록과 증강본 ID를 저장합니다.

**이 셀의 결과:** 공통 입력 쌍, 정답, 원본 분할 목록과 전처리 기록을 새 `run_...` 폴더에 저장한다. 같은 원본의 증강본이 검증·테스트에 들어가지 않도록 분할을 먼저 수행한다. 이 셀을 다시 실행하면 새 실험 폴더가 생기므로 이어서 4-A를 실행해야 한다.


In [ ]:
# 3 공통 데이터: 원본 분할 → 중복/전처리 검사 → 학습용 증강 → 두 표현을 함께 저장.
def frame_number(path):
    nums = re.findall(r'\d+',Path(path).stem)
    return int(nums[-2] if len(nums)>1 else nums[0]) if nums else 0

prepared = manifest.copy()
prepared['split'] = 'test'
for label in CLASS_NAMES:
    ids = prepared.index[(prepared['class']==label)&(prepared['official_split']=='Training')]
    part = prepared.loc[ids]
    prepared.loc[ids,'split'] = 'train'
    groups = part['source_folder'].unique()
    if len(groups)>1:
        val_ids = part.index[part['source_folder']==sorted(groups)[-1]]
    else:
        rotation = part['local_path'].map(lambda s: Path(s).stem.split('_')[0] if Path(s).stem.startswith('r') else 'plain')
        if rotation.nunique()>1:
            val_ids = rotation.index[rotation==sorted(rotation.unique())[-1]]
        else:
            ordered = sorted(ids, key=lambda i:frame_number(prepared.at[i,'local_path']))
            cut = max(1,int(len(ordered)*0.8))
            val_ids = ordered[cut:]
            boundary = frame_number(prepared.at[ordered[cut],'local_path'])
            gap_ids = [i for i in ordered[:cut] if frame_number(prepared.at[i,'local_path'])>=boundary-5]
            prepared.loc[gap_ids,'split']='excluded_gap'
    prepared.loc[val_ids,'split']='val'

# 두 표현을 같은 원본/마스크/증강본에서 동시에 생성
EXPERIMENT_DIR=CONTROLLED_ROOT/('run_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
EXPERIMENT_DIR.mkdir(parents=True,exist_ok=False)
prepared.to_csv(EXPERIMENT_DIR/'source_split.csv',index=False,encoding='utf-8-sig')
paired_lists={split:{'outline':[],'lineart':[],'y':[],'sample_ids':[]} for split in ['train','val','test']}
audit=[];sample_rows=[];seen_pixels=set();augmentation_rng=np.random.default_rng(SEED)

def append_pair(split,details,label,sample_id,source_path,view):
    outline=details['outline'].astype('float32')[...,None]/255
    lineart=details['lineart'].astype('float32')[...,None]/255
    if outline.shape!=(IMAGE_SIZE,IMAGE_SIZE,1) or outline.shape!=lineart.shape:
        raise ValueError('두 표현의 크기가 다릅니다.')
    paired_lists[split]['outline'].append(outline)
    paired_lists[split]['lineart'].append(lineart)
    paired_lists[split]['y'].append(label)
    paired_lists[split]['sample_ids'].append(sample_id)
    sample_rows.append({'sample_id':sample_id,'split':split,'source_path':source_path,
                        'class':CLASS_NAMES[label],'label':label,'augmentation_view':view})

for split in ['train','val','test']:
    for row in prepared[prepared['split']==split].to_dict('records'):
        try:
            rgb=read_rgb(row['local_path'])
            digest=photo_fingerprint(rgb)
            if digest in seen_pixels:raise ValueError('동일 원본 픽셀 중복 제외')
            _,details=extract_training_lineart(rgb)
            seen_pixels.add(digest)
            append_pair(split,details,int(row['label']),digest+'_v0',row['source_path'],0)
            added=0
            if split=='train':
                for view in range(1,AUGMENTED_VIEWS+1):
                    try:
                        aug_rgb,aug_mask=augment_training_object(details['rgb'],details['mask'],augmentation_rng)
                        _,aug_details=fruit_to_lineart(aug_rgb,aug_mask,IMAGE_SIZE,**LINEART_CONFIG)
                        append_pair(split,aug_details,int(row['label']),digest+f'_v{view}',row['source_path'],view)
                        added+=1
                    except ValueError:continue
            row.update(status='ok',error='',augmented_views=added,source_photo_id=digest)
        except (ValueError,OSError) as exc:
            row.update(status='excluded',error=str(exc),augmented_views=0)
        audit.append(row)

audit=pd.DataFrame(audit)
audit.to_csv(EXPERIMENT_DIR/'preprocessing_audit.csv',index=False,encoding='utf-8-sig')
pair_manifest=pd.DataFrame(sample_rows)
pair_manifest.to_csv(EXPERIMENT_DIR/'paired_sample_manifest.csv',index=False)
valid=audit[audit['status']=='ok']
counts=pd.crosstab(valid['class'],valid['split']).reindex(index=CLASS_NAMES,columns=['train','val','test'],fill_value=0).fillna(0)
display(counts)
if (counts<3).any().any():raise ValueError('각 종류/분할에 원본 사진이 최소 3장 필요합니다.')
paired_data={}
for split,part in paired_lists.items():
    order=np.random.default_rng(SEED).permutation(len(part['y']))
    paired_data[split]={
        'outline':np.stack(part['outline'])[order],
        'lineart':np.stack(part['lineart'])[order],
        'y':np.asarray(part['y'],dtype='int32')[order],
        'sample_ids':np.asarray(part['sample_ids'])[order]}
    assert paired_data[split]['outline'].shape==paired_data[split]['lineart'].shape
    print(split,'두 입력 공통 크기:',paired_data[split]['lineart'].shape)
# 학습 셀 밖의 기존 기능과도 호환되는 선화 별칭
x_train,y_train=paired_data['train']['lineart'],paired_data['train']['y']
x_val,y_val=paired_data['val']['lineart'],paired_data['val']['y']
x_test,y_test=paired_data['test']['lineart'],paired_data['test']['y']
np.savez_compressed(EXPERIMENT_DIR/'paired_test.npz',outline=paired_data['test']['outline'],
                    lineart=x_test,y=y_test,sample_ids=paired_data['test']['sample_ids'])
conditions={'classes':CLASS_NAMES,'image_size':IMAGE_SIZE,'mask_config':MASK_CONFIG,'lineart_config':LINEART_CONFIG,
    'split_seed':SEED,'training_seeds':EXPERIMENT_SEEDS,'epochs':EPOCHS,'batch_size':BATCH_SIZE,
    'augmented_views_per_training_original':AUGMENTED_VIEWS,'occlusion_probability':OCCLUSION_PROBABILITY,
    'max_occlusion_fraction':MAX_OCCLUSION_FRACTION,'source_commit':COMMIT,'source_folders':source_folders,
    'sample_count':{split:len(part['y']) for split,part in paired_data.items()},
    'sample_ids_sha256':{split:hashlib.sha256('\n'.join(part['sample_ids']).encode()).hexdigest() for split,part in paired_data.items()},
    'corrected_examples_included':False,'primary_inference':'single_0deg','secondary_inference':'rotation_mean_4'}
(EXPERIMENT_DIR/'conditions.json').write_text(json.dumps(conditions,ensure_ascii=False,indent=2),encoding='utf-8')
fig,axes=plt.subplots(len(CLASS_NAMES),3,figsize=(9,12))
for label,name in enumerate(CLASS_NAMES):
    row=valid[(valid['class']==name)&(valid['split']=='train')].iloc[0]
    _,details=extract_training_lineart(row['local_path'])
    for ax,key,title in zip(axes[label],['normalized_rgb','outline','drawing'],['Normalized photo','Outline only','Line art + details']):
        image=255-details[key] if key=='outline' else details[key]
        ax.imshow(image,cmap='gray',vmin=0,vmax=255);ax.set_title(f'{name}: {title}',fontsize=9);ax.axis('off')
plt.tight_layout();plt.savefig(EXPERIMENT_DIR/'paired_input_examples.png',dpi=160);plt.show()
print('공통 데이터 준비 완료:',EXPERIMENT_DIR)
print('교정 예제는 이 비교에 포함하지 않았습니다. 학습/검증/테스트 원본과 증강본이 두 모델에서 정확히 대응합니다.')


## 4-A. 같은 조건의 외곽선 CNN과 내부 특징 선화 CNN 학습
두 모델은 같은 구조·초기 가중치·배치 크기·에폭별 순서·고정 학습률을 사용합니다.
에폭 수를 모두 동일하게 채우고 각각 가장 낮은 검증 손실의 모델을 저장합니다. 선택 에폭이 달라도 선택 규칙과 총 학습 예산은 같습니다.
첫 실행은 두 모델을 새로 학습하므로 기존 모델을 불러오는 것보다 오래 걸립니다. 빠른 점검은 `TRAIN_PER_CLASS=60`, `TEST_PER_CLASS=20`, `EPOCHS=2`로 줄이세요.

**모델과 학습 설정:** Conv2D 16·32·64·96 → 각 ReLU·MaxPooling → Flatten → Dense 96 → Dropout 0.4 → 5종 Softmax. Adam, 학습률 0.0003, 배치 32, 기본 25 epochs를 사용한다. 손실은 `sparse_categorical_crossentropy`이며 학습 클래스 수 차이를 보정하는 클래스 가중치를 적용한다. 입력의 작은 평행 이동·확대축소도 두 모델에 공통 적용한다.

**선택 규칙:** 25회를 끝까지 학습하되, 중간에 검증 손실이 이전보다 낮아질 때 모델을 저장한다. 마지막 에폭의 가중치를 무조건 사용하지 않는다. 학습에는 증강·Dropout·클래스 가중치가 적용되므로 학습/검증 손실의 절대 크기를 그대로 비교하지 않고 전체 추이와 검증 최소점을 확인한다.

**주의:** 빠른 점검용으로 수량·에폭을 줄이면 PPT에 제시한 실험과 조건이 달라진다. 제출 결과를 재현하려면 기본 비교 조건을 유지한다.


In [ ]:
# 4-A: 같은 구조/초기 가중치/순서/고정 학습률/에폭으로 두 CNN 학습
def build_controlled_cnn(seed):
    tf.keras.utils.set_random_seed(seed)
    jitter=tf.keras.Sequential([
        layers.Input((IMAGE_SIZE,IMAGE_SIZE,1),name='jitter_input'),
        layers.RandomTranslation(0.04,0.04,fill_mode='constant',fill_value=0,seed=seed,name='translation'),
        layers.RandomZoom(0.05,fill_mode='constant',fill_value=0,seed=seed+1,name='zoom'),
    ],name='small_geometric_variations')
    network=tf.keras.Sequential([
        layers.Input((IMAGE_SIZE,IMAGE_SIZE,1),name='drawing'),jitter,
        layers.Conv2D(16,3,padding='same',activation='relu',name='conv1'),layers.MaxPooling2D(name='pool1'),
        layers.Conv2D(32,3,padding='same',activation='relu',name='conv2'),layers.MaxPooling2D(name='pool2'),
        layers.Conv2D(64,3,padding='same',activation='relu',name='conv3'),layers.MaxPooling2D(name='pool3'),
        layers.Conv2D(96,3,padding='same',activation='relu',name='conv4'),layers.MaxPooling2D(name='pool4'),
        layers.Flatten(name='flatten'),layers.Dense(96,activation='relu',name='features'),
        layers.Dropout(0.4,seed=seed+2,name='dropout'),layers.Dense(len(CLASS_NAMES),activation='softmax',name='classes'),
    ],name='controlled_fruit_cnn')
    network.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003,name='controlled_adam'),
                    loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return network

def weights_digest(weights):
    return hashlib.sha256(b''.join(np.ascontiguousarray(w).tobytes() for w in weights)).hexdigest()

if 'paired_data' not in globals():raise RuntimeError('먼저 3번 공통 데이터 준비 셀을 실행하세요.')
if not EXPERIMENT_SEEDS or len(set(EXPERIMENT_SEEDS))!=len(EXPERIMENT_SEEDS):raise ValueError('중복되지 않는 실험 시드를 지정하세요.')
if EPOCHS<1 or BATCH_SIZE<1:raise ValueError('에폭/배치 크기는 양수여야 합니다.')
tf.config.experimental.enable_op_determinism()
class_weights={i:float(len(y_train)/(len(CLASS_NAMES)*np.sum(y_train==i))) for i in range(len(CLASS_NAMES))}
training_records=[]
for run_seed in EXPERIMENT_SEEDS:
    seed_dir=EXPERIMENT_DIR/f'seed_{run_seed}';seed_dir.mkdir(exist_ok=True)
    template=build_controlled_cnn(run_seed)
    initial_weights=[w.copy() for w in template.get_weights()]
    initial_hash=weights_digest(initial_weights)
    architecture=template.to_json()
    epoch_orders=np.stack([np.random.default_rng(run_seed+epoch).permutation(len(y_train)) for epoch in range(EPOCHS)])
    np.save(seed_dir/'epoch_orders.npy',epoch_orders)
    orders_hash=hashlib.sha256(epoch_orders.tobytes()).hexdigest()
    (seed_dir/'architecture.json').write_text(architecture,encoding='utf-8')
    del template
    for variant in ['outline','lineart']:
        network=build_controlled_cnn(run_seed)
        network.set_weights(initial_weights)
        assert weights_digest(network.get_weights())==initial_hash
        assert network.to_json()==architecture
        train_input=paired_data['train'][variant];val_input=paired_data['val'][variant]
        model_file=seed_dir/f'{variant}_cnn.keras'
        history_dict={};best_loss=float('inf');best_epoch=0
        print(f'\n=== seed={run_seed}, input={variant}, fixed epochs={EPOCHS} ===',flush=True)
        started=time.perf_counter()
        for epoch,order in enumerate(epoch_orders):
            # 두 모델에 같은 에폭별 순서를 명시적으로 전달. 별도 shuffle 없음.
            h=network.fit(train_input[order],y_train[order],validation_data=(val_input,y_val),
                          initial_epoch=epoch,epochs=epoch+1,batch_size=BATCH_SIZE,
                          shuffle=False,class_weight=class_weights,verbose=2)
            for key,values in h.history.items():history_dict.setdefault(key,[]).extend(float(v) for v in values)
            loss=float(h.history['val_loss'][-1])
            if loss<best_loss:
                best_loss=loss;best_epoch=epoch+1;network.save(model_file)
        seconds=time.perf_counter()-started
        metadata={'representation':REPRESENTATION if variant=='lineart' else 'controlled_outline_v1',
            'input_variant':variant,'classes':CLASS_NAMES,'class_names_ko':CLASS_KO,'image_size':IMAGE_SIZE,
            'lineart_config':LINEART_CONFIG,'mask_config':MASK_CONFIG,'seed':run_seed,
            'source_commit':COMMIT,'source_folders':source_folders,'epochs_max':EPOCHS,'trained_epochs':EPOCHS,
            'best_epoch':best_epoch,'selection_rule':'minimum validation loss among fixed epochs',
            'optimizer':'Adam','learning_rate':0.0003,'batch_size':BATCH_SIZE,'parameter_count':network.count_params(),
            'initial_weights_sha256':initial_hash,'epoch_orders_sha256':orders_hash,
            'architecture_sha256':hashlib.sha256(architecture.encode()).hexdigest(),
            'dataset_audit_path':str(EXPERIMENT_DIR/'preprocessing_audit.csv'),
            'paired_sample_manifest':str(EXPERIMENT_DIR/'paired_sample_manifest.csv'),
            'controlled_run_dir':str(EXPERIMENT_DIR),'used_training_photo_ids':[],
            'corrected_examples_included':False,'tensorflow':tf.__version__,'opencv':cv2.__version__,
            'source':'https://github.com/fruits-360/fruits-360-100x100','license':'CC BY-SA 4.0'}
        (seed_dir/f'{variant}_config.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
        (seed_dir/f'{variant}_history.json').write_text(json.dumps(history_dict,indent=2),encoding='utf-8')
        training_records.append({'seed':run_seed,'input':variant,'trained_epochs':EPOCHS,'best_epoch':best_epoch,
            'best_val_loss':best_loss,'seconds':seconds,'parameters':network.count_params(),
            'initial_weights_sha256':initial_hash,'epoch_orders_sha256':orders_hash})
        del network
pd.DataFrame(training_records).to_csv(EXPERIMENT_DIR/'training_records.csv',index=False)
(EXPERIMENT_DIR/'training_complete.json').write_text(json.dumps({'seeds':EXPERIMENT_SEEDS,'inputs':['outline','lineart']}),encoding='utf-8')
CONTROLLED_REGISTRY.write_text(json.dumps({'run_dir':str(EXPERIMENT_DIR)},indent=2),encoding='utf-8')
# 시연에는 첫 번째 시드의 선화 CNN을 사용. 테스트 성능으로 시드를 선택하지 않음.
first_dir=EXPERIMENT_DIR/f'seed_{EXPERIMENT_SEEDS[0]}'
model=tf.keras.models.load_model(first_dir/'lineart_cnn.keras')
MODEL_METADATA=json.loads((first_dir/'lineart_config.json').read_text(encoding='utf-8'))
ACTIVE_MODEL_PATH=first_dir/'lineart_cnn.keras'
MODEL_TO_LOAD='controlled_lineart'
ensure_lineart_model()
display(pd.DataFrame(training_records))
print('두 모델의 조건을 동일하게 맞춰 학습했습니다. 4-B를 실행해 비교표와 그래프를 생성하세요.')


## 4-B. 보고서용 성능 비교표와 그래프
두 표현을 같은 테스트 원본에서 평가합니다. **주 비교는 외곽선/선화 모두 0도 예측**입니다.
정확도·macro F1·종류별 precision/recall/F1과 입력별 차이를 저장합니다. 4방향 평균 결과도 별도 행으로 제공합니다.
이미 학습을 끝냈다면 설정 → 2번 → 5번 → 이 셀만 실행해 보고서 그래프를 다시 만들 수 있습니다.

**출력 읽기:** `single_0deg`가 주 비교이고 `rotation_mean_4`는 보조 분석이다. 표의 지표는 0~1 값이며, 예를 들어 정확도 0.958은 95.8%이다. 두 정확도의 차이에 100을 곱한 값은 퍼센트포인트(%p)이다.

- Precision: 해당 과일이라고 예측한 것 중 맞힌 비율.
- Recall: 실제 해당 과일 중 맞힌 비율.
- F1: 과일별 Precision과 Recall의 조화평균. Macro F1은 과일별 F1을 평균하며, 평균 Precision·Recall을 다시 조화평균한 값과는 다를 수 있다.
- 혼동행렬: 행은 실제 정답, 열은 CNN 예측이며 대각선이 정답이다.

그래프에는 두 입력의 학습·검증 손실/정확도가 함께 표시된다. `report/test_predictions.csv`는 표본별 정답·예측과 5종 점수를 보존한다. PPT의 ROC-AUC는 이 저장 점수로 별도 계산한 Macro OvR AUC이다. AUC 0.9992를 정확도 99.92%로 해석하지 않는다.

**재실행 범위:** 이 셀은 모델을 재학습하지 않고 저장된 통제 비교 모델을 평가해 보고서 파일을 갱신한다. 테스트 결과를 보고 설정을 반복 선택했다면 새로운 독립 평가 자료로 다시 검증해야 한다.


In [ ]:
# 4-B 결과 생성: 저장된 두 CNN을 평가하며 재학습하지 않습니다.
# 4-B: 보고서용 통제 비교표/그래프 생성 (학습 완료 후 재실행 가능)
ensure_lineart_model()
if not CONTROLLED_REGISTRY.exists():raise FileNotFoundError('먼저 3번과 4-A에서 비교 모델을 학습하세요.')
run_dir=Path(json.loads(CONTROLLED_REGISTRY.read_text(encoding='utf-8'))['run_dir'])
conditions=json.loads((run_dir/'conditions.json').read_text(encoding='utf-8'))
if conditions['image_size']!=IMAGE_SIZE or conditions['lineart_config']!=LINEART_CONFIG:
    raise ValueError('학습 당시 설정과 현재 선화 설정이 다릅니다.')
with np.load(run_dir/'paired_test.npz',allow_pickle=False) as stored:
    eval_inputs={name:stored[name].copy() for name in ['outline','lineart']}
    eval_y=stored['y'].copy();eval_ids=stored['sample_ids'].copy()
if set(np.unique(eval_y))!=set(range(len(CLASS_NAMES))):raise ValueError('5종 모두 포함된 평가 자료가 필요합니다.')
REPORT_DIR=run_dir/'report';REPORT_DIR.mkdir(exist_ok=True)
metric_rows=[];class_rows=[];prediction_rows=[];first_predictions={};all_histories={}
for run_seed in conditions['training_seeds']:
    seed_dir=run_dir/f'seed_{run_seed}'
    pair_metadata=[json.loads((seed_dir/f'{v}_config.json').read_text(encoding='utf-8')) for v in ['outline','lineart']]
    for key in ['initial_weights_sha256','architecture_sha256','epoch_orders_sha256','trained_epochs','learning_rate','batch_size','parameter_count']:
        if pair_metadata[0][key]!=pair_metadata[1][key]:raise ValueError(f'두 모델 조건 불일치: {key}')
    for variant,metadata in zip(['outline','lineart'],pair_metadata):
        candidate=tf.keras.models.load_model(seed_dir/f'{variant}_cnn.keras')
        bundle=predict_lineart_scores(eval_inputs[variant],batch_size=BATCH_SIZE,prediction_model=candidate)
        all_histories[(run_seed,variant)]=json.loads((seed_dir/f'{variant}_history.json').read_text(encoding='utf-8'))
        for method,prob in [('single_0deg',bundle['single']),('rotation_mean_4',bundle['rotation_mean'])]:
            pred=prob.argmax(axis=1)
            result=classification_report(eval_y,pred,labels=range(len(CLASS_NAMES)),target_names=CLASS_NAMES,output_dict=True,zero_division=0)
            metric_rows.append({'seed':run_seed,'input':variant,'inference':method,'test_count':len(eval_y),
                'accuracy':float(np.mean(pred==eval_y)),'macro_f1':result['macro avg']['f1-score'],
                'macro_precision':result['macro avg']['precision'],'macro_recall':result['macro avg']['recall'],
                'weighted_f1':result['weighted avg']['f1-score'],'best_epoch':metadata['best_epoch'],
                'trained_epochs':metadata['trained_epochs'],'parameters':metadata['parameter_count']})
            for name in CLASS_NAMES:class_rows.append({'seed':run_seed,'input':variant,'inference':method,'class':name,**result[name]})
            for idx,sample_id in enumerate(eval_ids):
                prediction_rows.append({'sample_id':str(sample_id),'seed':run_seed,'input':variant,'inference':method,
                    'true_class':CLASS_NAMES[eval_y[idx]],'predicted_class':CLASS_NAMES[pred[idx]],
                    'correct':bool(pred[idx]==eval_y[idx]),**{f'score_{name}':float(prob[idx,k]) for k,name in enumerate(CLASS_NAMES)}})
            if run_seed==conditions['training_seeds'][0]:first_predictions[(variant,method)]=pred
        del candidate
metrics=pd.DataFrame(metric_rows);per_class=pd.DataFrame(class_rows)
primary=metrics[metrics['inference']=='single_0deg'].copy()
summary=metrics.groupby(['input','inference'])[['accuracy','macro_f1']].agg(['mean','std','count'])
class_table=per_class[per_class['inference']=='single_0deg'].groupby(['class','input'])[['precision','recall','f1-score']].mean().unstack('input').reindex(CLASS_NAMES)
display(primary.round(4));display(summary.round(4));display(class_table.round(4))
paired_accuracy=primary.pivot(index='seed',columns='input',values='accuracy')
paired_accuracy['lineart_minus_outline_pp']=100*(paired_accuracy['lineart']-paired_accuracy['outline'])
display(paired_accuracy.round(4))
print('주 비교는 같은 0도 예측입니다. 4방향 평균은 별도 요인으로 나누어 기록합니다.')
metrics.to_csv(REPORT_DIR/'all_metrics.csv',index=False)
primary.to_csv(REPORT_DIR/'primary_comparison.csv',index=False)
summary.to_csv(REPORT_DIR/'summary_mean_std.csv')
per_class.to_csv(REPORT_DIR/'per_class_metrics.csv',index=False)
class_table.to_csv(REPORT_DIR/'per_class_comparison.csv')
paired_accuracy.to_csv(REPORT_DIR/'accuracy_difference_pp.csv')
pd.DataFrame(prediction_rows).to_csv(REPORT_DIR/'test_predictions.csv',index=False)
first_seed=conditions['training_seeds'][0]
fig,axes=plt.subplots(1,2,figsize=(11,4))
for variant,color in [('outline','tab:blue'),('lineart','tab:orange')]:
    h=all_histories[(first_seed,variant)];epochs=range(1,len(h['loss'])+1)
    for ax,metric in zip(axes,['accuracy','loss']):
        ax.plot(epochs,h[metric],'--',color=color,label=f'{variant} train')
        ax.plot(epochs,h['val_'+metric],color=color,label=f'{variant} validation')
for ax,title in zip(axes,['Accuracy','Loss']):ax.set(xlabel='Epoch',title=f'{title} (seed {first_seed})');ax.legend(fontsize=8);ax.grid(alpha=0.2)
plt.tight_layout();plt.savefig(REPORT_DIR/'learning_curves.png',dpi=180);plt.show()
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,metric in zip(axes,['accuracy','macro_f1']):
    mean=primary.groupby('input')[metric].mean().reindex(['outline','lineart'])
    std=primary.groupby('input')[metric].std().reindex(['outline','lineart'])
    errors=std.values if len(conditions['training_seeds'])>1 else None
    ax.bar(mean.index,mean.values,yerr=errors,capsize=5 if errors is not None else 0,color=['tab:blue','tab:orange'])
    ax.set(ylim=(0,1.05),ylabel=metric,title=f'{metric}: identical 0-degree inference')
    for i,value in enumerate(mean.values):ax.text(i,value+0.015,f'{value:.1%}',ha='center')
plt.tight_layout();plt.savefig(REPORT_DIR/'accuracy_macro_f1.png',dpi=180);plt.show()
for method in ['single_0deg','rotation_mean_4']:
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,variant in zip(axes,['outline','lineart']):
        cm=confusion_matrix(eval_y,first_predictions[(variant,method)],labels=range(len(CLASS_NAMES)))
        ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(ax=ax,cmap='Blues',colorbar=False)
        ax.set_title(f'{variant}: {method}')
    plt.tight_layout();plt.savefig(REPORT_DIR/f'confusion_{method}.png',dpi=180);plt.show()
fig,ax=plt.subplots(figsize=(10,4))
recall=per_class[per_class['inference']=='single_0deg'].groupby(['class','input'])['recall'].mean().unstack('input').reindex(index=CLASS_NAMES,columns=['outline','lineart'])
recall.plot.bar(ax=ax,ylim=(0,1.05),rot=0);ax.set(ylabel='Recall',title='Per-class recall: same 0-degree inference')
ax.legend(loc='upper left',bbox_to_anchor=(1.01,1));plt.tight_layout();plt.savefig(REPORT_DIR/'per_class_recall.png',dpi=180);plt.show()
print('보고서용 결과 저장:',REPORT_DIR)
print('두 모델은 같은 에폭 수를 학습하고 각각 검증 손실이 가장 낮은 시점을 선택했습니다. 선택 에폭은 달라도 선택 규칙은 같습니다.')
print('시드가 1개면 반복 변동성을 판단할 수 없습니다. EXPERIMENT_SEEDS를 여러 개로 지정하면 평균/표준편차를 함께 기록합니다.')
print('이 비교에는 교정 예제가 없으며, 공개 분류 테스트 결과를 자동 탐지·분리 전체 정확도로 해석하지 않습니다.')


## 5. 사진 시연에 사용할 선화 모델 불러오기
기본 `latest`는 최초 모델·통제 비교 선화 모델·추가 학습 모델 중 가장 최근에 저장된 모델을 불러옵니다.
통제 비교 모델을 명시적으로 사용하려면 설정의 `MODEL_TO_LOAD=controlled_lineart`를 선택합니다. 테스트 성능으로 시드를 고르지 않고 첫 번째 시드를 사용합니다.
외곽선 CNN은 입력 표현이 다르므로 이 선화 시연에 자동으로 연결하지 않습니다. 4-B에서 별도로 정확하게 비교합니다.

**보고서의 비교 모델로 시연하려면:** 0-B 설정에서 `MODEL_TO_LOAD='controlled_lineart'`로 지정하고 이 셀을 실행한다. 기본 `latest`는 더 최근의 교정 모델이 있으면 그 모델을 선택할 수 있으므로 출력되는 **사용 모델 경로**를 확인한다. 이 선택은 4-B의 두 CNN 통제 비교 대상을 바꾸지 않는다.

5-A·5-B는 현재 불러온 선화 모델의 별도 분석이며, 사진 분석만 하려면 건너뛰고 6-A로 이동해도 된다.


In [ ]:
# 5 모델 선택: 아래 출력 경로로 통제 비교 모델인지 교정 모델인지 확인합니다.
# MODEL_TO_LOAD: latest / original / controlled_lineart
if MODEL_TO_LOAD not in ['latest','original','controlled_lineart']:raise ValueError('MODEL_TO_LOAD 설정을 확인하세요.')
original_pair=(RESULT_DIR/'fruit_lineart_cnn.keras',RESULT_DIR/'config.json')
candidates=[original_pair]
controlled_pair=None
if CONTROLLED_REGISTRY.exists():
    run=Path(json.loads(CONTROLLED_REGISTRY.read_text(encoding='utf-8'))['run_dir'])
    conditions=json.loads((run/'conditions.json').read_text(encoding='utf-8'))
    seed_dir=run/f"seed_{conditions['training_seeds'][0]}"
    controlled_pair=(seed_dir/'lineart_cnn.keras',seed_dir/'lineart_config.json')
    candidates.append(controlled_pair)
if MODEL_REGISTRY.exists():
    registered=json.loads(MODEL_REGISTRY.read_text(encoding='utf-8'))
    candidates.append((RESULT_DIR/registered['model_path'],RESULT_DIR/registered['config_path']))
if MODEL_TO_LOAD=='original':model_path,config_path=original_pair
elif MODEL_TO_LOAD=='controlled_lineart':
    if controlled_pair is None:raise FileNotFoundError('먼저 3번과 4-A에서 통제 비교 모델을 학습하세요.')
    model_path,config_path=controlled_pair
else:
    available=[pair for pair in candidates if pair[0].exists() and pair[1].exists()]
    if not available:raise FileNotFoundError('선화 모델이 없습니다. 먼저 1~4번을 실행하세요.')
    model_path,config_path=max(available,key=lambda pair:pair[0].stat().st_mtime)
if not model_path.exists() or not config_path.exists():raise FileNotFoundError('모델/설정 파일이 없습니다.')
saved=json.loads(config_path.read_text(encoding='utf-8'))
if saved.get('representation')!=REPRESENTATION or saved.get('classes')!=CLASS_NAMES:raise ValueError('현재 5종 선화 모델과 다릅니다.')
if saved.get('image_size')!=IMAGE_SIZE or saved.get('lineart_config')!=LINEART_CONFIG:raise ValueError('선화 크기/설정이 다릅니다.')
model=tf.keras.models.load_model(model_path);MODEL_METADATA=saved;ACTIVE_MODEL_PATH=model_path
ensure_lineart_model()
print('사용 모델:',ACTIVE_MODEL_PATH)
print('학습 데이터 분할 목록:',active_audit_path())
print('교정 학습에 포함된 사진 수:',len(MODEL_METADATA.get('used_training_photo_ids',[])))


## 선택 분석 안내

아래 5-A·5-B는 현재 불러온 선화 CNN의 회전 평균을 추가로 분석한다. 두 CNN의 핵심 비교는 앞의 4-B에서 완료된다. 사진만 분석하려면 6-A로 이동한다.


## 5-A. 비교용 테스트 선화 복원 (재학습 불필요)
저장된 전처리 목록에서 원래 `test`로 분리한 사진만 다시 읽어 공통 선화로 변환합니다. 학습·검증 사진이나 증강본을 평가에 섞지 않습니다.
같은 사진·같은 CNN·같은 선화로 기존 0도 방식과 4방향 점수 평균을 비교합니다. 일부 사진이 없으면 비교 대상을 바꾸지 않고 중단합니다.


In [ ]:
# 5-A: 이미 학습한 모델의 원래 공개 테스트 사진만 복원 (학습/검증 사진 제외)
ensure_lineart_model()
audit_path=active_audit_path()
if not audit_path.exists():
    raise FileNotFoundError('저장된 전처리 목록이 없습니다. 먼저 1~4번에서 학습하거나 해당 모델의 목록을 복원하세요.')
saved_audit=pd.read_csv(audit_path)
required={'split','status','class','label','source_path','local_path'}
if not required.issubset(saved_audit.columns): raise ValueError('테스트 목록 형식이 맞지 않습니다.')
test_manifest=saved_audit[(saved_audit['split']=='test')&(saved_audit['status']=='ok')].copy()
if test_manifest.empty: raise ValueError('원래 테스트 사진 목록이 비어 있습니다.')
test_inputs=[];test_targets=[];test_paths=[];failures=[]
for row in test_manifest.to_dict('records'):
    label=int(row['label'])
    if label<0 or label>=len(CLASS_NAMES) or row['class']!=CLASS_NAMES[label]:
        raise ValueError('테스트 목록의 정답 순서가 모델의 5종 순서와 다릅니다.')
    # 다른 PC로 프로젝트를 옮겨도 photos 폴더의 상대 경로로 찾을 수 있게 함
    photo_path=DATA_DIR/row['source_path']
    if not photo_path.exists(): photo_path=Path(row['local_path'])
    try:
        tensor,_=extract_training_lineart(photo_path)
        test_inputs.append(tensor);test_targets.append(label);test_paths.append(str(photo_path))
    except (ValueError,OSError) as exc:
        failures.append({'file':str(photo_path),'error':str(exc)})
if failures:
    display(pd.DataFrame(failures))
    raise RuntimeError('원래 테스트 사진 일부를 처리하지 못했습니다. 비교 대상을 임의로 줄이지 않고 중단합니다.')
x_compare=np.stack(test_inputs).astype('float32')
y_compare=np.asarray(test_targets,dtype='int32')
if set(np.unique(y_compare))!=set(range(len(CLASS_NAMES))):
    raise ValueError('5종 모두 포함된 테스트셋이 필요합니다.')
print('재학습 없이 같은 테스트 사진으로 비교합니다:',x_compare.shape)
display(pd.Series(y_compare).value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES))).to_frame('test_count'))


## 5-B. 5종 전체에서 기존 방식과 4방향 평균 성능 비교
전체 정확도·macro F1, 종류별 정답률(recall)·F1, 혼동행렬과 개선/악화 샘플 수를 비교합니다.
추가 방향 스트레스 표는 같은 원본의 방향을 바꾼 비교이며 독립된 새 테스트 사진이 늘어난 것은 아닙니다.
평균 방식이 성능을 떨어뜨린 경우도 표시합니다. 특정 딸기 사진 결과만으로 효과를 판단하지 마세요.


In [ ]:
# 5-B: 같은 CNN/테스트셋에서 기존 0도 vs 4방향 점수 평균 비교
ensure_lineart_model()
if 'x_compare' not in globals(): raise RuntimeError('먼저 5-A 테스트 사진 복원 셀을 실행하세요.')
EVAL_DIR=RESULT_DIR/'rotation_comparison'
EVAL_DIR.mkdir(parents=True,exist_ok=True)
comparison_bundle=predict_lineart_scores(x_compare,batch_size=BATCH_SIZE)
methods={'single_0deg':comparison_bundle['single'],
         'rotation_mean_4':comparison_bundle['rotation_mean']}
metric_rows=[];class_rows=[];reports={}
for method,prob in methods.items():
    pred=prob.argmax(axis=1)
    report=classification_report(y_compare,pred,labels=range(len(CLASS_NAMES)),
                                 target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    reports[method]=report
    metric_rows.append({'method':method,'test_count':len(y_compare),
                        'accuracy':float(np.mean(pred==y_compare)),
                        'macro_f1':report['macro avg']['f1-score'],
                        'weighted_f1':report['weighted avg']['f1-score']})
    for name in CLASS_NAMES:
        class_rows.append({'method':method,'class':name,**report[name]})
    pd.DataFrame(report).T.to_csv(EVAL_DIR/f'{method}_classification_report.csv')
comparison_metrics=pd.DataFrame(metric_rows)
per_class_metrics=pd.DataFrame(class_rows)
display(comparison_metrics.round(4))
class_comparison=per_class_metrics.pivot(index='class',columns='method',values=['recall','f1-score']).reindex(CLASS_NAMES)
class_comparison[('recall_delta_pp','rotation_mean_minus_single')]=100*(
    class_comparison[('recall','rotation_mean_4')]-class_comparison[('recall','single_0deg')])
display(class_comparison.round(4))
delta_pp=100*(comparison_metrics.iloc[1]['accuracy']-comparison_metrics.iloc[0]['accuracy'])
print(f'정확도 차이(평균 - 기존): {delta_pp:+.2f} 퍼센트포인트')
print('종류별 recall은 실제 그 과일 중 정답을 맞힌 비율입니다. 음수인 차이도 그대로 기록합니다.')

single_pred=methods['single_0deg'].argmax(axis=1)
mean_pred=methods['rotation_mean_4'].argmax(axis=1)
sample_comparison=pd.DataFrame({'file':test_paths,'true_class':[CLASS_NAMES[i] for i in y_compare],
    'single_class':[CLASS_NAMES[i] for i in single_pred],
    'rotation_mean_class':[CLASS_NAMES[i] for i in mean_pred],
    'single_correct':single_pred==y_compare,'rotation_mean_correct':mean_pred==y_compare})
sample_comparison['transition']=np.select([
    (single_pred!=y_compare)&(mean_pred==y_compare),
    (single_pred==y_compare)&(mean_pred!=y_compare),
    (single_pred==y_compare)&(mean_pred==y_compare)],
    ['fixed','regressed','both_correct'],default='both_wrong')
for method,prob in methods.items():
    for label,name in enumerate(CLASS_NAMES): sample_comparison[f'{method}_score_{name}']=prob[:,label]
print('결과 변화:');display(sample_comparison['transition'].value_counts().to_frame('count'))

# 추가 스트레스 비교: 같은 원본들의 4방향에서 기존 방식이 얼마나 달라지는지 확인
# 4개 회전본은 독립된 새 사진 4장이 아니므로 test_count와 구분해 기록
# 4방향 집합은 시작 방향을 90도 바꾸어도 같으므로 평균 예측은 공유할 수 있음
view_preds=comparison_bundle['by_rotation'].argmax(axis=2)
rotation_stress=pd.DataFrame({
    'rotation_degrees':[0,90,180,270],
    'single_accuracy_at_orientation':[(view_preds[:,k]==y_compare).mean() for k in range(4)],
    'four_rotation_mean_accuracy':[(mean_pred==y_compare).mean()]*4,
    'original_photo_count':[len(y_compare)]*4})
display(rotation_stress.round(4))
comparison_metrics.to_csv(EVAL_DIR/'comparison_metrics.csv',index=False)
per_class_metrics.to_csv(EVAL_DIR/'per_class_metrics.csv',index=False)
class_comparison.to_csv(EVAL_DIR/'per_class_comparison.csv')
sample_comparison.to_csv(EVAL_DIR/'comparison_predictions.csv',index=False,encoding='utf-8-sig')
rotation_stress.to_csv(EVAL_DIR/'rotation_stress.csv',index=False)
(EVAL_DIR/'comparison_settings.json').write_text(json.dumps({
    'classes':CLASS_NAMES,'representation':REPRESENTATION,'lineart_config':LINEART_CONFIG,
    'test_count':len(y_compare),'test_source':str(audit_path),'model_file':str(ACTIVE_MODEL_PATH),
    'model_sha256':hashlib.sha256(Path(ACTIVE_MODEL_PATH).read_bytes()).hexdigest(),
    'rotation_degrees':[0,90,180,270],'aggregation':'arithmetic_mean_of_softmax_scores',
    'test_scope':'Fruits-360 held-out classification; excludes automatic detection/segmentation errors'
},ensure_ascii=False,indent=2),encoding='utf-8')
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,(method,prob) in zip(axes,methods.items()):
    cm=confusion_matrix(y_compare,prob.argmax(axis=1),labels=range(len(CLASS_NAMES)))
    ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(method)
plt.tight_layout();plt.savefig(EVAL_DIR/'comparison_confusion.png',dpi=150);plt.show()
fig,ax=plt.subplots(figsize=(10,4))
recall_table=per_class_metrics.pivot(index='class',columns='method',values='recall').reindex(index=CLASS_NAMES,columns=['single_0deg','rotation_mean_4'])
recall_table.plot.bar(ax=ax,ylim=(0,1),rot=0)
ax.set(ylabel='Per-class recall',title='Same five-class test set: 0 degrees vs rotation mean')
ax.legend(loc='upper left',bbox_to_anchor=(1.01,1))
plt.tight_layout();plt.savefig(EVAL_DIR/'per_class_recall_comparison.png',dpi=150);plt.show()
print('비교 결과 저장:',EVAL_DIR)
print('4방향 평균은 분류 계산량이 대략 4배입니다. 탐지·분리는 한 번만 수행합니다.')
print('성능 향상을 보장하지 않으며, 공개 테스트 결과를 복잡한 실사진 전체 정확도로 해석하지 마세요.')


## 6-A. 복잡한 사진의 자동 탐지·분리 준비
아래 셀은 함수를 준비합니다. 모델은 6-B에서 사진을 선택한 뒤 최초 1회 내려받아 메모리에 유지합니다.
`BOX_THRESHOLD`를 낮추면 후보가 늘어나는 대신 잘못된 탐지가 늘 수 있습니다. 모델이 놓친 과일은 CNN까지 전달되지 않습니다.

| 모델 | 입력 | 출력과 역할 |
|---|---|---|
| Grounding DINO Tiny | 원본 사진 + 5종 과일 텍스트 | 과일 후보의 위치 박스를 찾음 |
| SlimSAM | 원본 사진 + 탐지 박스 | 과일 픽셀 영역의 마스크를 만듦 |

두 모델은 추가 학습하지 않는다. 분리한 영역을 공통 선화 함수에 전달하고, 최종 과일 종류는 CNN으로 다시 분류한다. 탐지 점수·마스크 품질 추정값·CNN 점수는 서로 다른 값이다. 겹침·가림·잘림에서는 누락, 중복 탐지 또는 부정확한 마스크가 생길 수 있다.


In [ ]:
# 6-A: 자동 과일 탐지 및 개별 분리 준비 (이 셀에서는 사진 선택 창을 열지 않음)
import inspect, uuid
from datetime import datetime

DETECTOR_ID = 'IDEA-Research/grounding-dino-tiny'
SEGMENTER_ID = 'Zigeng/SlimSAM-uniform-77'
DETECTION_TEXT = 'an apple. a banana. a lemon. a strawberry. a pineapple.'
BOX_THRESHOLD = 0.28
TEXT_THRESHOLD = 0.25
BOX_NMS_IOU = 0.65
MASK_NMS_IOU = 0.80
MAX_OBJECTS = 20
MAX_PHOTO_SIDE = 1024
MIN_MASK_PIXELS = 25
MIN_SAM_SCORE = 0.50  # SAM의 마스크 품질 추정값. 정답 확률은 아님
AUTO_CACHE = ROOT / 'model_cache'
SCENE_DIR = RESULT_DIR / 'automatic_scenes'
AUTO_CACHE.mkdir(parents=True, exist_ok=True)
SCENE_DIR.mkdir(parents=True, exist_ok=True)
_SCENE_MODELS = None

def load_scene_models():
    global _SCENE_MODELS
    if _SCENE_MODELS is not None: return _SCENE_MODELS
    # Transformers에서 Keras를 자동으로 로드하지 않도록 PyTorch만 사용
    os.environ['USE_TF'] = '0'
    os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
    try:
        import torch
        import transformers
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor
    except (ImportError, OSError) as exc:
        raise RuntimeError('자동 분리 라이브러리를 불러오지 못했습니다. 설치 셀 실행 후 커널을 재시작하세요.') from exc
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('자동 분리 모델 준비:', device, '| 첫 실행은 모델 다운로드로 오래 걸릴 수 있습니다.', flush=True)
    kw = {'cache_dir':str(AUTO_CACHE)}
    try:
        detector_processor = AutoProcessor.from_pretrained(DETECTOR_ID, **kw)
        detector = AutoModelForZeroShotObjectDetection.from_pretrained(
            DETECTOR_ID, use_safetensors=True, **kw).to(device).eval()
        sam_processor = SamProcessor.from_pretrained(SEGMENTER_ID, **kw)
        sam = SamModel.from_pretrained(SEGMENTER_ID, use_safetensors=True, **kw).to(device).eval()
    except Exception as exc:
        raise RuntimeError('사전학습 모델 로딩 실패. 인터넷 연결, 디스크 공간, 설치 버전을 확인하세요. 잘못된 흰 배경 방식으로 대신 예측하지 않습니다.') from exc
    _SCENE_MODELS = (torch, device, detector_processor, detector, sam_processor, sam)
    (SCENE_DIR/'model_versions.json').write_text(json.dumps({
        'detector':DETECTOR_ID, 'segmenter':SEGMENTER_ID,
        'detector_revision':getattr(detector.config,'_commit_hash',None),
        'segmenter_revision':getattr(sam.config,'_commit_hash',None),
        'transformers':transformers.__version__, 'torch':torch.__version__, 'device':device
    },indent=2),encoding='utf-8')
    return _SCENE_MODELS

def box_iou(a,b):
    a,b = np.asarray(a,dtype=float),np.asarray(b,dtype=float)
    wh = np.maximum(0, np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2]))
    inter = float(wh[0]*wh[1])
    area_a = float(np.prod(np.maximum(0,a[2:]-a[:2])))
    area_b = float(np.prod(np.maximum(0,b[2:]-b[:2])))
    return inter/max(area_a+area_b-inter,1e-9)

def suppress_boxes(boxes,scores,threshold=0.65):
    kept=[]
    for idx in np.argsort(-np.asarray(scores)):
        if all(box_iou(boxes[idx],boxes[j]) <= threshold for j in kept): kept.append(int(idx))
    return kept

def mask_iou(a,b):
    a,b = np.asarray(a,dtype=bool),np.asarray(b,dtype=bool)
    return float(np.logical_and(a,b).sum()/max(np.logical_or(a,b).sum(),1))

def suppress_group_masks(objects):
    # 여러 개별 후보를 거의 통째로 포함하는 큰 합성 마스크를 제외하는 휴리스틱
    kept=[]; removed=[]
    for i,item in enumerate(objects):
        large=item['mask']; area=int(large.sum()); contained=[]
        for j,other in enumerate(objects):
            if i==j: continue
            small=other['mask']; small_area=int(small.sum())
            coverage=np.logical_and(large,small).sum()/max(small_area,1)
            if area>1.8*small_area and coverage>0.85:
                contained.append(j)
        has_distinct_pair=any(mask_iou(objects[a]['mask'],objects[b]['mask'])<0.25
                              for pos,a in enumerate(contained) for b in contained[pos+1:])
        if has_distinct_pair: removed.append(item)
        else: kept.append(item)
    return kept,removed

def detect_and_segment_fruits(source):
    torch,device,dp,detector,sp,sam = load_scene_models()
    rgb=read_rgb(source)
    h,w=rgb.shape[:2]
    scale=min(1.0,MAX_PHOTO_SIDE/max(h,w))
    if scale<1: rgb=cv2.resize(rgb,(round(w*scale),round(h*scale)),interpolation=cv2.INTER_AREA)
    h,w=rgb.shape[:2]
    image=Image.fromarray(rgb)
    print('1/3: 과일 후보 위치 자동 탐지 중...',flush=True)
    inputs=dp(images=image,text=DETECTION_TEXT,return_tensors='pt').to(device)
    with torch.inference_mode(): outputs=detector(**inputs)
    post=dp.post_process_grounded_object_detection
    threshold_name='box_threshold' if 'box_threshold' in inspect.signature(post).parameters else 'threshold'
    detected=post(outputs,inputs.input_ids,text_threshold=TEXT_THRESHOLD,
                  target_sizes=[(h,w)],**{threshold_name:BOX_THRESHOLD})[0]
    boxes=detected['boxes'].detach().cpu().numpy().reshape(-1,4)
    scores=detected['scores'].detach().cpu().numpy()
    labels=detected['text_labels'] if 'text_labels' in detected else detected.get('labels',['']*len(boxes))
    boxes[:,[0,2]]=boxes[:,[0,2]].clip(0,w)
    boxes[:,[1,3]]=boxes[:,[1,3]].clip(0,h)
    valid=np.flatnonzero((boxes[:,2]-boxes[:,0]>=3)&(boxes[:,3]-boxes[:,1]>=3))
    indices=[int(valid[i]) for i in suppress_boxes(boxes[valid],scores[valid],BOX_NMS_IOU)]
    if len(indices)>MAX_OBJECTS:
        print(f'후보 {len(indices)}개 중 점수가 높은 {MAX_OBJECTS}개만 처리합니다.')
        indices=indices[:MAX_OBJECTS]
    if not indices:
        return rgb,[],[{'status':'no_detection','reason':'설정한 5종의 과일 후보를 찾지 못했습니다.'}]
    print(f'2/3: 후보 {len(indices)}개 영역 분리 중...',flush=True)
    base=sp(images=image,return_tensors='pt').to(device)
    with torch.inference_mode(): embeddings=sam.get_image_embeddings(base['pixel_values'])
    accepted=[]; audit=[]
    for number,idx in enumerate(indices,1):
        box=boxes[idx].tolist()
        prompt=sp(images=image,input_boxes=[[box]],return_tensors='pt').to(device)
        with torch.inference_mode():
            sam_output=sam(image_embeddings=embeddings,input_boxes=prompt['input_boxes'],multimask_output=True)
        restored=sp.image_processor.post_process_masks(sam_output.pred_masks.cpu(),
                    prompt['original_sizes'].cpu(),prompt['reshaped_input_sizes'].cpu())[0][0]
        quality=sam_output.iou_scores.detach().cpu().numpy()[0,0]
        best=int(np.argmax(quality))
        mask=restored[best].numpy().astype(bool)
        # SAM이 박스 바깥의 다른 물체까지 포함하는 경우를 줄임
        x1,y1,x2,y2=box
        px=max(2,int((x2-x1)*0.08)); py=max(2,int((y2-y1)*0.08))
        allowed=np.zeros((h,w),bool)
        allowed[max(0,int(y1)-py):min(h,int(np.ceil(y2))+py),max(0,int(x1)-px):min(w,int(np.ceil(x2))+px)]=True
        mask &= allowed
        diagnostic={'candidate':number,'detector_hint':str(labels[idx]),'detector_score':float(scores[idx]),
                    'sam_score':float(quality[best]),'box':box}
        try:
            if quality[best] < MIN_SAM_SCORE: raise ValueError('분리 모델의 마스크 품질 점수가 낮음')
            tensor,details=fruit_to_lineart(rgb,mask,IMAGE_SIZE,**LINEART_CONFIG)
            binary=details['mask']>0
            if binary.sum()<MIN_MASK_PIXELS: raise ValueError('분리 영역이 너무 작음')
            if binary.mean()>0.90: raise ValueError('사진 대부분을 차지하는 잘못된 분리 영역')
            if any(mask_iou(binary,item['mask'])>MASK_NMS_IOU for item in accepted):
                raise ValueError('다른 후보와 중복된 마스크')
            accepted.append({'mask':binary,'tensor':tensor,'outline':details['outline'],'lineart':details['lineart'],
                             'normalized_mask':details['normalized_mask'],'diagnostic':diagnostic})
            audit.append({**diagnostic,'status':'accepted','reason':''})
        except ValueError as exc:
            audit.append({**diagnostic,'status':'rejected','reason':str(exc)})
        print(f'  영역 처리 {number}/{len(indices)}',flush=True)
    accepted,groups=suppress_group_masks(accepted)
    for item in groups:
        candidate=item['diagnostic']['candidate']
        for record in audit:
            if record.get('candidate')==candidate:
                record.update(status='rejected_group_mask',reason='여러 개별 후보를 포함한 큰 합성 영역')
    return rgb,accepted,audit

def classify_drawings(tensors):
    # 최종 예측은 선화 4방향 점수의 평균. RGB/탐지 라벨은 입력하지 않음
    return predict_lineart_scores(tensors,batch_size=BATCH_SIZE)['rotation_mean']

print('자동 분리 함수 준비 완료. 6-B 셀을 실행하면 사진 선택 창이 바로 열립니다.')


## 6-B. 사진 선택 → 여러 과일 분리 → 내부 특징 선화 → 분류
**아래 셀을 실행하면 즉시 사진 선택 창이 열립니다.** 사진 선택 후 과일을 자동 분리하고 공통 함수로 선화를 만들어 CNN에 입력합니다.
화면에서 분리된 과일, 외곽선 비교 그림, 내부 특징 선화, CNN 결과를 확인할 수 있습니다. 다른 사진은 같은 셀만 재실행하세요.
새 모델을 학습/불러오지 않았으면 먼저 재학습 안내가 나옵니다. 사진의 가려진 부분을 복원하거나 모든 과일을 반드시 맞히는 기능은 아닙니다.

최종 분류는 0·90·180·270도 softmax 점수의 평균입니다. 기존 0도 결과와 평균 결과를 나란히 표시하고, `rotation_scores.csv`에 과일별·방향별 5종 점수를 저장합니다. 같은 사진의 과일을 다시 탐지하거나 분리하지 않습니다.

**현재 기능의 기본 동작:** 이 사진 분석 셀은 기존 기능대로 **4방향 평균을 최종 결과로 표시**하며 단일 예측도 함께 제공한다. PPT의 핵심 CNN 비교는 4-B의 단일 예측이므로 두 값을 혼동하지 않는다. Streamlit의 추론 방식 선택과도 별개이다. 평균 방식이 5종 전체 성능을 항상 높이는 것은 아니다.

사진을 고를 때 정답은 입력하지 않는다. 새 사진을 분석하려면 **6-B 코드 셀만 다시 실행**한다. 창이 안 보이면 작업 표시줄 또는 Alt+Tab을 확인하고, 취소했다면 다시 실행한다. CNN 점수는 모델 출력값이며 실제 정답 확률을 보장하지 않는다.


In [ ]:
# 6-B: 실행하면 즉시 사진 선택 창 → 자동 분리 → 컴퓨터 선화 생성 → 선화 CNN 분류
def predict_photo(source,title='My photo',show=True):
    global LAST_SCENE_DIR
    ensure_lineart_model()
    if 'detect_and_segment_fruits' not in globals():
        raise RuntimeError('먼저 6-A 자동 분리 준비 셀을 실행하세요.')
    rgb,objects,audit=detect_and_segment_fruits(source)
    run_dir=SCENE_DIR/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
    run_dir.mkdir(parents=True,exist_ok=False)
    LAST_SCENE_DIR=run_dir
    photo_id=photo_fingerprint(rgb)
    used_in_training=photo_id in MODEL_METADATA.get('used_training_photo_ids',[])
    if used_in_training: print('주의: 이 사진은 교정 학습에 포함된 사진입니다. 새 사진 평가 결과로 사용하지 마세요.')
    Image.fromarray(rgb).save(run_dir/'photo.png')
    pd.DataFrame(audit).to_csv(run_dir/'segmentation_audit.csv',index=False,encoding='utf-8-sig')
    (run_dir/'settings.json').write_text(json.dumps({
        'source_name':title,'classes':CLASS_NAMES,'representation':REPRESENTATION,'lineart_config':LINEART_CONFIG,'mask_config':MASK_CONFIG,'image_size':IMAGE_SIZE,
        'detector':DETECTOR_ID,'segmenter':SEGMENTER_ID,'query':DETECTION_TEXT,
        'box_threshold':BOX_THRESHOLD,'text_threshold':TEXT_THRESHOLD,'box_nms_iou':BOX_NMS_IOU,
        'mask_nms_iou':MASK_NMS_IOU,'sam_quality_threshold':MIN_SAM_SCORE,'max_objects':MAX_OBJECTS,
        'max_photo_side':MAX_PHOTO_SIDE,'classifier_input':'lineart_only',
        'prediction_mode':'rotation_mean_4','rotation_degrees':[0,90,180,270],
        'aggregation':'arithmetic_mean_of_softmax_scores',
        'source_photo_id':photo_id,'used_in_feedback_training':used_in_training,
        'active_model_file':str(ACTIVE_MODEL_PATH)
    },ensure_ascii=False,indent=2),encoding='utf-8')
    if not objects:
        print('분리된 과일이 없습니다. 탐지·분리 로그를 확인하세요.')
        display(pd.DataFrame(audit))
        if show:
            plt.figure(figsize=(8,5));plt.imshow(rgb);plt.axis('off');plt.show()
        table=pd.DataFrame(columns=['object_id','cnn_class','cnn_name_ko','cnn_score'])
        table.to_csv(run_dir/'predictions.csv',index=False)
        print('결과 폴더:',run_dir)
        return table
    print(f'3/3: 컴퓨터가 그린 선화 {len(objects)}개를 CNN으로 분류 중...',flush=True)
    score_bundle=predict_lineart_scores([obj['tensor'] for obj in objects],batch_size=BATCH_SIZE)
    scores=score_bundle['rotation_mean']
    rows=[];rotation_rows=[]
    for i,(obj,prob) in enumerate(zip(objects,scores),1):
        mask=obj['mask']; best=int(prob.argmax())
        single_prob=score_bundle['single'][i-1]; single_best=int(single_prob.argmax())
        ys,xs=np.where(mask)
        left,top,right,bottom=int(xs.min()),int(ys.min()),int(xs.max())+1,int(ys.max())+1
        crop=rgb[top:bottom,left:right]
        alpha=(mask[top:bottom,left:right]*255).astype('uint8')
        rgba=np.dstack([crop,alpha])
        obj['rgba']=rgba
        # 흰 바탕의 검은 선 = 사람이 볼 자동 그림 / CNN 입력은 그 반전
        drawing=255-obj['lineart']
        Image.fromarray(mask.astype('uint8')*255).save(run_dir/f'object_{i:02d}_mask.png')
        Image.fromarray(rgba).save(run_dir/f'object_{i:02d}_cutout.png')
        Image.fromarray(drawing).save(run_dir/f'object_{i:02d}_drawing.png')
        Image.fromarray(obj['lineart']).save(run_dir/f'object_{i:02d}_cnn_input.png')
        Image.fromarray(255-obj['outline']).save(run_dir/f'object_{i:02d}_outline_only.png')
        rows.append({'object_id':i,'cnn_class':CLASS_NAMES[best],'cnn_name_ko':CLASS_KO[best],
                     'cnn_score':float(prob[best]),
                     'single_class':CLASS_NAMES[single_best],'single_name_ko':CLASS_KO[single_best],
                     'single_score':float(single_prob[single_best]),
                     'changed_by_rotation_mean':bool(single_best!=best),'left':left,'top':top,'right':right,'bottom':bottom,
                     'review_note':('사진 경계에 닿은 부분 윤곽: 오분류에 주의' if (mask[0].any() or mask[-1].any() or mask[:,0].any() or mask[:,-1].any()) else ''),
                     'drawing_file':str(run_dir/f'object_{i:02d}_drawing.png'),
                     **{f'cnn_score_{c}':float(prob[k]) for k,c in enumerate(CLASS_NAMES)}})
        for k,angle in enumerate([0,90,180,270]):
            direction_prob=score_bundle['by_rotation'][i-1,k]
            rotation_rows.append({'object_id':i,'rotation_degrees':angle,
                'predicted_class':CLASS_NAMES[int(direction_prob.argmax())],
                **{f'score_{name}':float(direction_prob[j]) for j,name in enumerate(CLASS_NAMES)}})
    table=pd.DataFrame(rows)
    pd.DataFrame(rotation_rows).to_csv(run_dir/'rotation_scores.csv',index=False)
    table.to_csv(run_dir/'predictions.csv',index=False,encoding='utf-8-sig')
    if show:
        from matplotlib.patches import Rectangle
        fig,ax=plt.subplots(figsize=(10,6))
        ax.imshow(rgb)
        for row in rows:
            x,y,x2,y2=row['left'],row['top'],row['right'],row['bottom']
            ax.add_patch(Rectangle((x,y),x2-x,y2-y,fill=False,color='lime',linewidth=2))
            ax.text(x,y,f"#{row['object_id']} CNN mean: {row['cnn_class']} ({row['cnn_score']:.0%})",
                    color='white',fontsize=9,bbox={'facecolor':'black','alpha':0.65,'pad':2})
        ax.set_title(f'{title}: line-art CNN: four-rotation mean');ax.axis('off')
        fig.tight_layout();fig.savefig(run_dir/'overview.png',dpi=140);plt.show()
        for i,(obj,prob) in enumerate(zip(objects,scores),1):
            fig,axes=plt.subplots(1,4,figsize=(12,3))
            axes[0].imshow(obj['rgba']);axes[0].set_title(f'#{i}: segmented fruit')
            axes[1].imshow(255-obj['outline'],cmap='gray',vmin=0,vmax=255);axes[1].set_title('Outline only (comparison)')
            axes[2].imshow(255-obj['lineart'],cmap='gray',vmin=0,vmax=255);axes[2].set_title('Line art: contour + details')
            positions=np.arange(len(CLASS_NAMES))
            axes[3].barh(positions-0.18,score_bundle['single'][i-1],height=0.35,label='0 degrees')
            axes[3].barh(positions+0.18,prob,height=0.35,label='4-rotation mean')
            axes[3].set_yticks(positions,CLASS_NAMES);axes[3].set_xlim(0,1)
            axes[3].set_title('Drawing-only CNN');axes[3].legend(fontsize=7)
            for ax in axes[:3]: ax.axis('off')
            fig.tight_layout();fig.savefig(run_dir/f'object_{i:02d}_steps.png',dpi=140);plt.show()
        formatted=table[['object_id','single_name_ko','single_score','cnn_name_ko','cnn_score','review_note']].copy()
        formatted['cnn_score']=formatted['cnn_score'].map(lambda x:f'{x:.1%}')
        formatted['single_score']=formatted['single_score'].map(lambda x:f'{x:.1%}')
        display(formatted)
        print('single은 기존 0도, cnn은 0/90/180/270도 점수 평균입니다. 탐지 이름은 최종 분류에 사용하지 않습니다.')
        print('누락·겹침·잘못된 마스크와 낮은 분류 성능은 화면의 중간 결과를 함께 확인하세요.')
    print('자동 그림과 결과 저장:',run_dir)
    return table

def choose_and_predict_photo():
    if 'model' not in globals() or 'fruit_to_lineart' not in globals():
        raise RuntimeError('설정 → 2번 공통 함수 → 5번 모델 불러오기 → 6-A 셀을 먼저 실행하세요.')
    if 'detect_and_segment_fruits' not in globals(): raise RuntimeError('6-A 셀을 먼저 실행하세요.')
    ensure_lineart_model()
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError as exc:
        raise RuntimeError('tkinter가 포함된 로컬 Windows Python 커널을 선택하세요.') from exc
    window=None
    print('사진 선택 창을 엽니다. 안 보이면 작업 표시줄 또는 Alt+Tab을 확인하세요.',flush=True)
    try:
        window=tk.Tk();window.withdraw();window.attributes('-topmost',True);window.update()
        filename=filedialog.askopenfilename(parent=window,title='과일을 자동 분석할 사진 선택',
            filetypes=[('사진 파일','*.jpg *.jpeg *.png *.bmp *.webp'),('모든 파일','*.*')])
    except tk.TclError as exc:
        raise RuntimeError('파일 선택 창을 열지 못했습니다. tkinter가 포함된 로컬 Windows Python 커널인지 확인하세요.') from exc
    finally:
        if window is not None: window.destroy()
    if not filename:
        print('선택을 취소했습니다. 다시 선택하려면 이 셀을 실행하세요.')
        return None
    print('선택한 사진:',filename)
    return predict_photo(filename,Path(filename).name)

choose_and_predict_photo()


## 선택 확장 안내: 정답 교정과 추가 학습

7-A·7-B는 사람이 확인한 실사진 선화를 수집하고 추가 학습하는 기능이다. 필수 보고서 비교에는 사용하지 않았으며, 제출 확인을 위해 반드시 실행할 필요는 없다.


## 7-A. 잘못된 예측의 실제 정답 확인 및 저장
**이 셀을 실행하면 직전 분석 사진의 정답 선택 창이 열립니다.** 사진 선택 창과 같은 로컬 tkinter 창이라 ipywidgets가 필요 없습니다.
과일 번호와 분리 이미지/선화를 보고 실제 정답을 고릅니다. 맞았던 과일도 확인해서 저장할 수 있으며, 미선택 항목은 저장하지 않습니다. 분리가 틀린 영역은 제외합니다.
사진 전체 용도를 학습용·검증용·평가용 중 선택합니다. 같은 사진의 과일을 서로 다른 용도로 나눌 수 없습니다. 평가용은 추가 학습이나 에폭 선택에 사용하지 않습니다.
같은 파일의 중복은 묶지만, 같은 실물을 다른 각도로 촬영한 사진까지 자동으로 알아내지는 못합니다. 새로운 평가에는 가능하면 다른 실제 과일을 사용하세요.
저장은 예측 결과를 덮어쓰지 않습니다. **저장만으로 모델의 답이 바뀌지 않으며 7-B를 실행해야 합니다.**

**여기서는 새 사진 파일을 고르지 않는다.** 6-B에서 방금 분석한 사진의 과일별 정답을 확인하는 단계이다. 핵심 비교 실험과 일반 사진 예측에는 이 단계를 실행하지 않아도 된다. 저장된 정답은 학습 자료이며, 예측할 때 정답을 조회해 답을 대신 출력하는 기능이 아니다.


In [ ]:
# 7-A: 직전 사진의 과일별 정답을 사람이 확인하여 저장
import shutil,uuid
from datetime import datetime
from PIL import ImageTk

def feedback_config_key():
    config={'representation':REPRESENTATION,'classes':CLASS_NAMES,'image_size':IMAGE_SIZE,'lineart_config':LINEART_CONFIG}
    return hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()

def read_feedback():
    if FEEDBACK_INDEX.exists(): return json.loads(FEEDBACK_INDEX.read_text(encoding='utf-8'))
    return {'schema':1,'groups':{},'samples':{}}

def atomic_json(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix('.tmp')
    temp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
    temp.replace(path)

def save_reviewed_labels(run_dir,answers,split='train'):
    """answers={과일번호: 정답 class 또는 'exclude'}. 예측은 정답으로 자동 저장하지 않음."""
    run_dir=Path(run_dir)
    if split not in ['train','validation','test']: raise ValueError('저장 용도를 확인하세요.')
    settings=json.loads((run_dir/'settings.json').read_text(encoding='utf-8'))
    if (settings.get('representation')!=REPRESENTATION or settings.get('classes')!=CLASS_NAMES or
        settings.get('lineart_config')!=LINEART_CONFIG or settings.get('image_size')!=IMAGE_SIZE):
        raise ValueError('현재 선화 설정으로 처리한 사진만 교정에 사용할 수 있습니다.')
    predictions=pd.read_csv(run_dir/'predictions.csv')
    if predictions.empty: raise ValueError('분리된 과일이 없습니다.')
    valid_ids=set(predictions['object_id'].astype(int))
    answers={int(k):v for k,v in answers.items()}
    if not answers: return 0
    if not set(answers).issubset(valid_ids): raise ValueError('과일 번호를 확인하세요.')
    if not all(v in CLASS_NAMES+['exclude'] for v in answers.values()): raise ValueError('정답은 5종 또는 제외만 가능합니다.')
    photo=read_rgb(run_dir/'photo.png');group_id=photo_fingerprint(photo)
    data=read_feedback()
    if group_id in data['groups'] and data['groups'][group_id]['split']!=split:
        raise ValueError('같은 사진의 과일은 용도를 바꿔 학습/검증/평가에 나눠 넣을 수 없습니다.')
    # 모든 파일을 확인한 뒤 기록: 분리 오류를 정답 과일로 강제로 학습시키지 않음
    pending=[]
    for object_id,label in answers.items():
        mask_path=run_dir/f'object_{object_id:02d}_mask.png'
        input_path=run_dir/f'object_{object_id:02d}_cnn_input.png'
        with Image.open(input_path) as im: pixels=np.asarray(im.convert('L'))
        if pixels.shape!=(IMAGE_SIZE,IMAGE_SIZE): raise ValueError('선화 크기가 다릅니다.')
        if not np.isin(pixels,[0,255]).all(): raise ValueError('이진 선화 파일이 아닙니다.')
        mask_hash=hashlib.sha256(mask_path.read_bytes()).hexdigest()
        sample_id=hashlib.sha256((group_id+mask_hash+feedback_config_key()).encode()).hexdigest()
        pending.append((object_id,label,sample_id))
    photo_dir=FEEDBACK_DIR/'photos'/group_id
    photo_dir.mkdir(parents=True,exist_ok=True)
    shutil.copy2(run_dir/'photo.png',photo_dir/'photo.png')
    now=datetime.now().isoformat(timespec='seconds')
    data['groups'][group_id]={'split':split,'source_name':settings.get('source_name',''),
                             'photo_file':str(photo_dir/'photo.png'),'updated_at':now}
    for object_id,label,sample_id in pending:
        folder=FEEDBACK_DIR/'examples'/sample_id;folder.mkdir(parents=True,exist_ok=True)
        for suffix in ['cnn_input','drawing','cutout','mask']:
            shutil.copy2(run_dir/f'object_{object_id:02d}_{suffix}.png',folder/f'{suffix}.png')
        original=predictions[predictions['object_id']==object_id].iloc[0]
        record={'sample_id':sample_id,'photo_id':group_id,'split':split,
            'true_class':None if label=='exclude' else label,'status':'excluded' if label=='exclude' else 'labeled',
            'config_key':feedback_config_key(),'input_file':str(folder/'cnn_input.png'),
            'drawing_file':str(folder/'drawing.png'),'cutout_file':str(folder/'cutout.png'),
            'source_run':str(run_dir),'source_object_id':object_id,
            'original_prediction':str(original['cnn_class']),'original_score':float(original['cnn_score']),
            'updated_at':now,'label_source':'human_review'}
        old=data['samples'].get(sample_id)
        if old:
            record['history']=old.get('history',[])+[{'true_class':old.get('true_class'),'status':old.get('status'),'at':old.get('updated_at')}]
        data['samples'][sample_id]=record
    atomic_json(FEEDBACK_INDEX,data)
    return len(pending)

def show_feedback_counts():
    records=[r for r in read_feedback()['samples'].values() if r['status']=='labeled' and r['config_key']==feedback_config_key()]
    if not records: print('아직 저장한 정답이 없습니다.');return
    frame=pd.DataFrame(records)
    display(pd.crosstab(frame['true_class'],frame['split']).reindex(CLASS_NAMES,fill_value=0))
    print('서로 다른 사진 수:',frame.groupby('split')['photo_id'].nunique().to_dict())

def open_correction_window(run_dir=None):
    import tkinter as tk
    from tkinter import ttk,filedialog,messagebox
    window=tk.Tk();window.withdraw()
    try:
        if run_dir is None:
            selected=filedialog.askdirectory(parent=window,title='분석 결과 폴더 선택',initialdir=str(SCENE_DIR))
            if not selected: return
            run_dir=Path(selected)
        run_dir=Path(run_dir)
        predictions=pd.read_csv(run_dir/'predictions.csv')
        if predictions.empty: print('교정할 과일이 없습니다.');return
        group_id=photo_fingerprint(read_rgb(run_dir/'photo.png'))
        data=read_feedback();existing_group=data['groups'].get(group_id)
        window.title('과일 정답 확인 및 학습 예제 저장');window.geometry('880x700')
        ttk.Label(window,text='예측과 별개로 실제 정답을 선택하세요. 확인하지 않은 항목은 저장하지 않습니다.').pack(padx=10,pady=8)
        purpose_options={'학습용(train)':'train','검증용(validation)':'validation','새 사진 평가용(test)':'test'}
        purpose=tk.StringVar(value=next((k for k,v in purpose_options.items() if existing_group and v==existing_group['split']),'학습용(train)'))
        purpose_frame=ttk.Frame(window);purpose_frame.pack(fill='x',padx=10)
        ttk.Label(purpose_frame,text='이 사진 전체의 용도:').pack(side='left')
        purpose_box=ttk.Combobox(purpose_frame,textvariable=purpose,values=list(purpose_options),state='readonly',width=26)
        purpose_box.pack(side='left',padx=8)
        if existing_group: purpose_box.configure(state='disabled')
        ttk.Label(window,text='같은 사진의 모든 과일은 같은 용도입니다. 분리 실패/과일 아닌 영역은 제외하세요.').pack(pady=5)
        body=ttk.Frame(window);body.pack(fill='both',expand=True,padx=10)
        canvas=tk.Canvas(body,highlightthickness=0);bar=ttk.Scrollbar(body,orient='vertical',command=canvas.yview)
        canvas.configure(yscrollcommand=bar.set);bar.pack(side='right',fill='y');canvas.pack(side='left',fill='both',expand=True)
        rows=ttk.Frame(canvas);canvas.create_window((0,0),window=rows,anchor='nw')
        rows.bind('<Configure>',lambda event:canvas.configure(scrollregion=canvas.bbox('all')))
        choices={'선택 안 함':None,**dict(zip(CLASS_KO,CLASS_NAMES)),'제외: 과일 아님/분리 실패':'exclude'}
        variables={};image_refs=[]
        settings=json.loads((run_dir/'settings.json').read_text(encoding='utf-8'))
        for row_idx,row in enumerate(predictions.to_dict('records')):
            oid=int(row['object_id']);frame=ttk.Frame(rows);frame.pack(fill='x',pady=6)
            ttk.Label(frame,text=f'#{oid} 예측: {row.get("cnn_name_ko",row["cnn_class"])} ({row["cnn_score"]:.0%})',width=25).pack(side='left')
            for suffix in ['cutout','drawing']:
                with Image.open(run_dir/f'object_{oid:02d}_{suffix}.png') as image:
                    thumb=image.copy();thumb.thumbnail((130,100))
                rendered=ImageTk.PhotoImage(thumb,master=window);image_refs.append(rendered)
                ttk.Label(frame,image=rendered).pack(side='left',padx=5)
            variable=tk.StringVar(value='선택 안 함');variables[oid]=variable
            # 예측값은 기본 정답으로 채우지 않음. 이전에 사람이 확인한 정답만 복원
            matching=[r for r in data['samples'].values() if r['photo_id']==group_id and r['config_key']==feedback_config_key()
                      and Path(r['source_run'])==run_dir and r['source_object_id']==oid]
            if matching:
                label='exclude' if matching[-1]['status']=='excluded' else matching[-1]['true_class']
                variable.set(next(k for k,v in choices.items() if v==label))
            ttk.Combobox(frame,textvariable=variable,values=list(choices),state='readonly',width=23).pack(side='left',padx=5)
        footer=ttk.Frame(window);footer.pack(fill='x',padx=10,pady=10)
        def commit():
            answers={oid:choices[var.get()] for oid,var in variables.items() if choices[var.get()] is not None}
            try:
                count=save_reviewed_labels(run_dir,answers,purpose_options[purpose.get()])
            except (ValueError,OSError) as exc:
                messagebox.showerror('저장 실패',str(exc),parent=window);return
            messagebox.showinfo('저장 완료',f'{count}개 저장했습니다. 저장만으로 예측이 바뀌지는 않습니다. 추가 학습 셀을 실행하세요.',parent=window)
            window.destroy()
        ttk.Button(footer,text='선택한 정답 저장',command=commit).pack(side='right')
        ttk.Button(footer,text='취소',command=window.destroy).pack(side='right',padx=8)
        window.deiconify();window.lift();window.mainloop()
    finally:
        try:window.destroy()
        except tk.TclError:pass
    show_feedback_counts()

open_correction_window(globals().get('LAST_SCENE_DIR'))


## 7-B. 교정 선화를 기존 5종 데이터와 섞어 추가 학습
사람이 확인한 학습용 선화와, 원래 학습용으로 분리한 5종 사진을 섞어 낮은 학습률로 추가 학습합니다. 특정 과일만 배워 다른 과일을 잊는 현상을 줄이기 위해 5종 균형을 맞춥니다.
검증 손실로 에폭을 선택하고, 공개 테스트 및 별도로 남긴 새 사진 평가용 자료에서 전후 결과를 표시합니다. **교정 학습 사진을 다시 맞힌 비율은 성능 평가에 넣지 않습니다.**
모델을 새 버전으로 저장하므로 최초 모델은 보관됩니다. 추가 학습이 끝나면 새 버전이 현재 세션과 다음 실행의 기본 모델이 됩니다.
새 사진을 고를 때 정답은 입력하지 않습니다. 정답 교정 창은 학습 예제를 만드는 별도 단계입니다. 최근 모델이 학습한 사진을 다시 예측하면 학습 사진이라는 안내가 나옵니다.

**추가 학습은 선택 사항이다.** 새 사진에서 좋아졌는지는 학습·검증에 사용하지 않은 별도 사진으로 판단한다. 교정 사진을 다시 맞혔다고 일반화가 개선됐다고 결론 내리지 않는다. 비교 실험을 그대로 유지하려면 이 단계 대신 4-B 결과를 사용한다.


In [ ]:
# 7-B: 사람이 확인한 학습용 선화 + 기존 5종 학습 사진으로 추가 학습
FINETUNE_EPOCHS=8
REPLAY_PER_CLASS=100
PUBLIC_VAL_PER_CLASS=60
CORRECTION_WEIGHT=3.0

def load_official_records(split,limit_per_class=None):
    frame=pd.read_csv(active_audit_path())
    frame=frame[(frame['split']==split)&(frame['status']=='ok')].copy()
    if limit_per_class:
        frame=pd.concat([group.sample(n=min(len(group),limit_per_class),random_state=SEED)
                         for _,group in frame.groupby('class',sort=False)],ignore_index=True)
    return frame

def official_tensors(frame):
    xs=[];ys=[];paths=[]
    for row in frame.to_dict('records'):
        path=DATA_DIR/row['source_path']
        if not path.exists():path=Path(row['local_path'])
        tensor,_=extract_training_lineart(path)
        xs.append(tensor);ys.append(int(row['label']));paths.append(str(path))
    if not xs:raise ValueError('공개 데이터 목록이 비어 있습니다.')
    return np.stack(xs),np.asarray(ys,dtype='int32'),paths

def correction_tensors(records):
    xs=[];ys=[]
    for record in records:
        with Image.open(record['input_file']) as image:pixels=np.asarray(image.convert('L'))
        if pixels.shape!=(IMAGE_SIZE,IMAGE_SIZE):raise ValueError('교정 선화 크기가 다릅니다.')
        xs.append(pixels.astype('float32')[...,None]/255)
        ys.append(CLASS_NAMES.index(record['true_class']))
    if not xs:return np.empty((0,IMAGE_SIZE,IMAGE_SIZE,1),np.float32),np.empty(0,np.int32)
    return np.stack(xs),np.asarray(ys,dtype='int32')

def fit_reviewed_examples():
    global model,MODEL_METADATA,ACTIVE_MODEL_PATH,MODEL_TO_LOAD
    ensure_lineart_model()
    records=[r for r in read_feedback()['samples'].values() if r['status']=='labeled' and r['config_key']==feedback_config_key()]
    train_records=[r for r in records if r['split']=='train']
    val_records=[r for r in records if r['split']=='validation']
    test_records=[r for r in records if r['split']=='test']
    if not train_records:
        print('학습용으로 저장한 정답이 없습니다. 6-B → 7-A에서 먼저 저장하세요.');return None
    train_groups={r['photo_id'] for r in train_records}
    heldout_groups={r['photo_id'] for r in val_records+test_records}
    inherited_groups=set(MODEL_METADATA.get('used_training_photo_ids',[]))
    if (train_groups|inherited_groups)&heldout_groups:
        raise ValueError('교정 학습 사진과 검증/평가 사진이 겹칩니다. 사진 단위 분리를 확인하세요.')
    # 공개 검증/테스트 사진을 교정 학습용으로 다시 넣은 경우도 차단
    official_audit=pd.read_csv(active_audit_path())
    forbidden=official_audit[(official_audit['split'].isin(['val','test']))&(official_audit['status']=='ok')]
    forbidden_hashes=set()
    for row in forbidden.to_dict('records'):
        photo=DATA_DIR/row['source_path']
        if not photo.exists():photo=Path(row['local_path'])
        forbidden_hashes.add(photo_fingerprint(read_rgb(photo)))
    if train_groups&forbidden_hashes:
        raise ValueError('공개 검증/테스트와 동일한 사진이 교정 학습에 들어 있습니다. 평가 사진은 학습에 사용하지 않습니다.')
    replay_frame=load_official_records('train',REPLAY_PER_CLASS)
    validation_frame=load_official_records('val',PUBLIC_VAL_PER_CLASS)
    replay_x,replay_y,replay_paths=official_tensors(replay_frame)
    val_x,val_y,_=official_tensors(validation_frame)
    if set(replay_y)!=set(range(len(CLASS_NAMES))):raise ValueError('기존 5종 학습 사진이 모두 필요합니다.')
    feedback_x,feedback_y=correction_tensors(train_records)
    # 정확한 90도 회전은 학습용 교정 선화에만 추가
    expanded_x=np.concatenate([np.rot90(feedback_x,k,axes=(1,2)) for k in range(4)])
    expanded_y=np.tile(feedback_y,4)
    train_x=np.concatenate([replay_x,expanded_x]).astype('float32')
    train_y=np.concatenate([replay_y,expanded_y])
    weight=np.concatenate([np.ones(len(replay_y)),np.full(len(expanded_y),CORRECTION_WEIGHT)]).astype('float32')
    # 한 종류만 교정해도 그 종류가 전체 학습을 압도하지 않도록 5종 균형 유지
    for label in range(len(CLASS_NAMES)):
        ids=train_y==label
        weight[ids]*=len(weight)/(len(CLASS_NAMES)*weight[ids].sum())
    feedback_val_x,feedback_val_y=correction_tensors(val_records)
    if len(feedback_val_y):val_x=np.concatenate([val_x,feedback_val_x]);val_y=np.concatenate([val_y,feedback_val_y])
    candidate=tf.keras.models.clone_model(model);candidate.set_weights(model.get_weights())
    candidate.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                      loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    version_dir=FINETUNE_DIR/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
    version_dir.mkdir(parents=True,exist_ok=False)
    candidate_path=version_dir/'fruit_lineart_cnn.keras'
    callbacks=[tf.keras.callbacks.ModelCheckpoint(str(candidate_path),monitor='val_loss',save_best_only=True),
               tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)]
    print('교정 학습 사진:',len(train_groups),'| 정답 선화:',len(train_records),'| 기존 5종 복습:',len(replay_y))
    print('교정한 사진을 다시 맞히는 것은 일반화 성능 평가가 아닙니다.')
    history=candidate.fit(train_x,train_y,sample_weight=weight,validation_data=(val_x,val_y),
                          epochs=FINETUNE_EPOCHS,batch_size=BATCH_SIZE,callbacks=callbacks,verbose=2)
    candidate=tf.keras.models.load_model(candidate_path)
    # 테스트 지표로 모델/에폭을 고르지 않고, 검증 손실로 고른 후보의 결과만 기록
    public_test_x,public_test_y,_=official_tensors(load_official_records('test'))
    evaluations=[('public_test',public_test_x,public_test_y)]
    reserved_x,reserved_y=correction_tensors(test_records)
    if len(reserved_y):evaluations.append(('new_photo_reserved_test',reserved_x,reserved_y))
    metric_rows=[]
    for dataset,x,y in evaluations:
        present=np.unique(y)
        for name,network in [('before',model),('after',candidate)]:
            prob=predict_lineart_scores(x,batch_size=BATCH_SIZE,prediction_model=network)['rotation_mean']
            pred=prob.argmax(axis=1)
            report=classification_report(y,pred,labels=present,output_dict=True,zero_division=0)
            metric_rows.append({'dataset':dataset,'model':name,'count':len(y),
                'classes_present':','.join(CLASS_NAMES[i] for i in present),
                'accuracy':float(np.mean(pred==y)),'macro_f1_present_classes':report['macro avg']['f1-score']})
    metrics=pd.DataFrame(metric_rows);display(metrics.round(4))
    if not test_records:print('새 사진 평가용 정답이 없습니다. 공개 테스트 점검만 수행했으므로 실사진 성능 개선은 아직 입증하지 못했습니다.')
    metrics.to_csv(version_dir/'before_after_metrics.csv',index=False)
    used=pd.DataFrame(train_records);used.to_csv(version_dir/'feedback_training_manifest.csv',index=False,encoding='utf-8-sig')
    validation_frame.to_csv(version_dir/'public_validation_manifest.csv',index=False)
    replay_frame.to_csv(version_dir/'public_replay_manifest.csv',index=False)
    (version_dir/'history.json').write_text(json.dumps(history.history),encoding='utf-8')
    metadata={**MODEL_METADATA,'parent_model_file':str(ACTIVE_MODEL_PATH),
        'parent_model_sha256':hashlib.sha256(Path(ACTIVE_MODEL_PATH).read_bytes()).hexdigest(),
        'used_training_photo_ids':sorted(inherited_groups|train_groups),
        'feedback_sample_ids':sorted(set(MODEL_METADATA.get('feedback_sample_ids',[]))|{r['sample_id'] for r in train_records}),
        'finetune_epochs_max':FINETUNE_EPOCHS,'finetune_learning_rate':1e-5,
        'human_labeling':'labels used only for training/validation/evaluation; not a prediction lookup'}
    config_path=version_dir/'config.json';atomic_json(config_path,metadata)
    atomic_json(MODEL_REGISTRY,{'model_path':os.path.relpath(candidate_path,RESULT_DIR),
                               'config_path':os.path.relpath(config_path,RESULT_DIR)})
    model=candidate;MODEL_METADATA=metadata;ACTIVE_MODEL_PATH=candidate_path
    MODEL_TO_LOAD='latest'
    ensure_lineart_model()
    print('추가 학습 모델을 현재 예측에 사용합니다:',ACTIVE_MODEL_PATH)
    print('최초 모델로 돌아가려면 설정의 MODEL_TO_LOAD를 original로 바꾸고 5번을 실행하세요.')
    print('동일한 교정 학습 사진을 다시 넣으면 학습에 사용한 사진임을 표시합니다.')
    return metrics

feedback_finetuning_results=fit_reviewed_examples()


## 8. 결과 파일과 제출 결과 확인

프로젝트의 최상위 폴더는 0-B에서 출력한 `fruit_outline_project`이다.

| 내용 | 저장 위치 |
|---|---|
| 다운로드한 원본·목록 | `photos/`, `metadata/` |
| 두 CNN 통제 비교 | `results/controlled_outline_vs_lineart_v1/run_.../` |
| 비교 모델·선택 에폭·학습 이력 | 위 실험 폴더의 `seed_42/`, `training_records.csv` |
| 공통 분할·표본·전처리 | `source_split.csv`, `paired_sample_manifest.csv`, `preprocessing_audit.csv`, `conditions.json` |
| 전체·종류별 지표와 그래프 | 위 실험 폴더의 `report/` |
| 사진별 자동 분리·선화·예측 | `results/lineart_v1_apple_banana_lemon_strawberry_pineapple/automatic_scenes/실행시간_고유번호/` |
| 현재 선화 모델의 별도 회전 분석 | 위 선화 모델 폴더의 `rotation_comparison/` |
| 사람이 확인한 정답 자료 | `reviewed_lineart/` |
| 교정 후 새 모델·전후 지표 | 위 선화 모델 폴더의 `finetuned_versions/` |

`*_drawing.png`는 흰 바탕·검은 선의 보기용 그림, `*_cnn_input.png`는 검은 바탕·흰 선의 CNN 입력, `*_outline_only.png`는 외곽선 비교용 그림이다. 통제 비교에서는 외곽선과 내부 특징 선화가 각각 별도 CNN의 입력이다.

### PPT와 연결되는 저장 실험

PPT의 결과는 `run_20261104_121829_f10503`, seed 42의 기록이다. 같은 조건이라도 실행 환경이나 새 데이터 버전에 따라 재실행 값이 완전히 같지 않을 수 있다. 아래 값은 새 실행 결과를 대신하는 출력이 아니라 **기존 저장 실험의 요약**이다.

| 주 비교: 0도 단일 예측 | 외곽선 | 내부 특징 선화 |
|---|---:|---:|
| 정확도 | 84.40% | 95.80% |
| Macro Precision | 85.37% | 95.94% |
| Macro Recall | 84.40% | 95.80% |
| Macro F1 | 0.8423 | 0.9574 |
| Macro OvR ROC-AUC (저장 점수의 별도 후처리) | 0.9682 | 0.9992 |
| 검증 손실 최소 에폭 | 21 | 11 |

학습 원본 1,350장, 증강 포함 학습 입력 4,050장, 검증 900장, 테스트 500장이다. 테스트는 과일별 100장이다. 선화의 4방향 평균 정확도는 94.0%로 단일 예측보다 낮았다. 선화 CNN은 마지막 25회가 아니라 검증 손실이 최소인 11회 가중치를 평가했다.

### 실험에서 확인한 점과 남은 한계

1. 외곽선 대비 내부 특징 선화의 공개 테스트 성능은 높았지만, 개별 오분류 원인을 모두 확정한 것은 아니다.
2. 특정 딸기 사진의 개선만으로 회전 평균의 전체 효과를 판단할 수 없었다. 전체 테스트에서 개선이 없었던 결과도 기록한다.
3. 학습 후반의 검증 손실 증가를 확인하고 최소 검증 손실 모델을 선택했다. 이것만으로 실제 사진의 일반화 문제가 해결됐다고 주장하지 않는다.
4. 과일 개체 수준의 독립성, 복잡한 배경·겹침·단면 사진 성능은 추가 검증이 필요하다. seed 42 한 번의 비교이며 손그림 인식 성능도 별도로 평가하지 않았다.

### 출처와 활용 모델

- Mihai Oltean, [Fruits-360 100×100](https://github.com/fruits-360/fruits-360-100x100), [CC BY-SA 4.0](https://github.com/fruits-360/fruits-360-100x100/blob/main/LICENSE). 일부 사진을 선택해 외곽선·선화로 가공했다. 데이터 버전은 `metadata/version.json`과 실험 설정에 기록한다.
- [Grounding DINO Tiny](https://huggingface.co/IDEA-Research/grounding-dino-tiny), [SlimSAM](https://huggingface.co/Zigeng/SlimSAM-uniform-77): 사전학습 탐지·분리 모델, Apache-2.0.
- [OpenCV 외곽선 처리](https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html), [Canny](https://docs.opencv.org/4.11.0/da/d5c/tutorial_canny_detector.html), [Transformers SAM](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/sam).

### 자주 생기는 실행 문제

| 증상 | 확인할 사항 |
|---|---|
| `manifest`가 없다는 오류 | 1번 다운로드 셀을 먼저 실행. 저장 모델 시연만 할 때는 2번 미리보기 생략 |
| 모델·설정·전처리 목록을 찾지 못함 | 해당 프로젝트 결과 폴더가 있는지 확인. 처음 실행하거나 다른 PC라면 1~4번 수행 |
| 사진 선택 창이 안 보임 | 로컬 Python·tkinter 사용 여부, 작업 표시줄·Alt+Tab 확인 |
| 다운로드가 오래 걸리거나 API 제한 발생 | 최초 다운로드·네트워크 상태 확인. 캐시가 있으면 재사용 |
| 과일을 놓치거나 엉뚱하게 분류 | 원본 → 탐지 박스 → 분리 마스크 → 선화 → CNN 점수를 순서대로 확인 |

제출 확인은 핵심 비교 결과와 선택 기능을 구분해서 진행한다. 사진 선택·정답 교정 창을 자동으로 건너뛰거나 저장된 정답을 예측 답으로 사용하지 않는다.
